# HTGAA CFPS — reproducible clustering and pre-run selection audit

**Repository context:** [RonanChance/htgaa-automation](https://github.com/RonanChance/htgaa-automation) — the existing SvelteKit/PocketBase application and source of truth for the CFPS designer.

This notebook is intended to be committed later at `notebooks/cfps_database_snapshot_and_exploration.ipynb` in that repository. It does **not** push commits automatically.

## Source-aligned details verified in the repository

- PocketBase base URL: `https://opentrons-art-pb.rcdonovan.com`.
- Existing server code authenticates through PocketBase's `_superusers` collection using `PB_EMAIL` and `PB_PASSWORD`. The previous draft's `users` default was not aligned with the repository and is corrected here.
- Community formulation submissions are stored in `cfps_designs`; each record's `design_json` contains `sample_id`, `sample_type`, `reagent_list`, and `metadata` (including the target protein and batch fields where present).
- The reagent catalogue is stored in `cfps_reagent_groups`, record `t7a97pyfs8lx67w`.
- The repo's CFPS contract refers to reagent volumes in nL, a 25 nL grid, and a 20 µL reaction total. The notebook reports volume totals as a diagnostic; it does not discard records that differ from the current contract.

Relevant source files: [`src/lib/server/pb.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/lib/server/pb.js), [`src/lib/server/loadCommunityDesigns.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/lib/server/loadCommunityDesigns.js), [`src/lib/server/loadCfpsReagentGroups.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/lib/server/loadCfpsReagentGroups.js), and [`src/routes/cfps/submit-design/+server.js`](https://github.com/RonanChance/htgaa-automation/blob/main/src/routes/cfps/submit-design/+server.js).

## Security / data handling

This notebook performs **read-only GET requests**, apart from the authentication POST required to obtain a token. It does not submit, update, or delete CFPS designs. Credentials are loaded from Colab Secrets or hidden prompts and are never written to files or printed. The Keycloak client secret is not needed for this direct PocketBase flow and is intentionally not used.

Raw records (which may contain authorship, comments, conversation links, or other member-associated data) are saved only under `/content/cfps_private_snapshots/`, **outside the cloned Git repository**. The `reports/cfps_exploration/` directory inside the repo receives aggregate tables and figures only. Review sharing/redistribution permission with the coordinator before committing even derived outputs. Do not commit raw snapshots or row-level exports.

**Access condition:** the `PB_EMAIL` / `PB_PASSWORD` pair must be credentials authorized to authenticate against `_superusers` on this PocketBase instance, as the repository's server code does. If Ronan supplied ordinary Keycloak login credentials rather than PB superuser credentials, do not try to work around access controls; ask him for the intended approved access route.

**Two independent data paths:** optional read-only PocketBase snapshot/recipe-pattern correlation (Sections 1–12); and the published GP-informed uploaded `clusters_designs.json` pre-run audit (Sections 13–16). The second path requires no credentials, live database, or installed local `.py` modules. All row-level results are private by default. The compiled notebook embeds the exact module source; it does not import or create the mirrored `.py` modules during its analysis.

**Research attribution (methods only):** The GP-informed forecasts, shared-error clustering, virtual-LHS sensitivity analysis and original proposed runs were developed by **Clinton Monteiro**. The model-free recipe-space auditing, deduplication, exact-contrast analysis, facility-location and Maximin selection, and comparison visualization were developed by **Kourosh Afshinjoo**. The `htgaa-automation` platform, authentication and deployment infrastructure are maintained by **Ronan Chance**. Scientific methods are otherwise named by their mathematical purpose, not their contributors. See `docs/CONTRIBUTIONS.md`.

**Run contract:** Choose `INPUT_SOURCE` in the configuration cell. Each run records the original input SHA-256 and a normalized snapshot SHA-256. PocketBase-only runs cannot reconstruct the GP-informed comparison without the independently supplied, version-matched annotated export. Private ID-level exports remain outside Git.


In [ ]:
# Source selection: change ONE variable before running the notebook.
# "published_snapshot"   : upload clusters_designs.json; includes GP-informed annotations.
# "pocketbase_snapshot"  : upload a previously fetched pocketbase_snapshot.json.
# "pocketbase_live"      : fetch the current read-only snapshot using authorized credentials.
INPUT_SOURCE = "published_snapshot"
PUBLISHED_SNAPSHOT_PATH = "/content/clusters_designs.json"
POCKETBASE_SNAPSHOT_PATH = "/content/pocketbase_snapshot.json"
GP_LINKS_PATH = "/content/clusters.json.gz"  # optional; relevant to published snapshot only
EXTERNAL_PLATES_ZIP = "/content/cfps-plates.zip"  # optional
EXTERNAL_POLICY = "flag"  # "flag" or "exclude"; never silently discard an ID
WORKING_QUOTA_NOTEBOOK = 320  # unverified working assumption
GEOMETRY_OBJECTIVE = "facility"  # facility | maximin; both modes evaluated by default
AUTO_DOWNLOAD_PRIVATE_RESULTS = True

RUN_POCKETBASE_FETCH = INPUT_SOURCE == "pocketbase_live"
RUN_CORRELATION_CLUSTERING = False  # optional independent descriptive analysis
RUN_GP_BASELINE_AUDIT = INPUT_SOURCE == "published_snapshot"
RUN_PORTFOLIO_PREVIEWS = False
RUN_MODEL_FREE_PIPELINE = True
GP_DESIGNS_PATH = PUBLISHED_SNAPSHOT_PATH  # backward-compatible base-audit input
print("Input source:", INPUT_SOURCE)
print("GP-informed comparison available:", RUN_GP_BASELINE_AUDIT)


In [ ]:
# Environment (Colab). GP-export-only analysis does not clone GitHub or request credentials.
from pathlib import Path
import os, sys, subprocess
REPO_URL = "https://github.com/RonanChance/htgaa-automation.git"
REPO_ROOT = Path("/content/htgaa-automation")
PRIVATE_ROOT = Path("/content/cfps_private_snapshots")
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_POCKETBASE_FETCH:
    if not (REPO_ROOT / ".git").exists():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError(f"{REPO_ROOT} exists but is not a Git checkout")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
    REPO_SHA = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
    REPORT_DIR = PRIVATE_ROOT / "aggregate_review"
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
else:
    REPO_SHA = None
    REPORT_DIR = PRIVATE_ROOT / "aggregate_review"
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas", "scipy", "requests", "matplotlib"], check=True)
print("Read-only repository reference:", REPO_URL, "(cloned only in PocketBase mode)")
print("Private analysis folder:", PRIVATE_ROOT)

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 2. Load credentials from Colab Secrets, otherwise prompt without echoing the password.
    # Recommended Colab secret names: PB_EMAIL and PB_PASSWORD. Optional: PB_URL.
    import os
    from getpass import getpass

    try:
        from google.colab import userdata
    except Exception:
        userdata = None

    def secret_or_prompt(key, prompt, hidden=True):
        value = None
        if userdata is not None:
            try:
                value = userdata.get(key)
            except Exception:
                value = None
        if not value:
            value = getpass(prompt) if hidden else input(prompt)
        if not value:
            raise ValueError(f"No value provided for {key}.")
        os.environ[key] = value

    # Prefer an authorized PocketBase authentication token when available.
    # The platform's Keycloak agent token is NOT automatically a PocketBase JWT.
    PB_TOKEN = os.environ.get("PB_TOKEN", "")
    if userdata is not None and not PB_TOKEN:
        try:
            PB_TOKEN = userdata.get("PB_TOKEN") or ""
        except Exception:
            PB_TOKEN = ""
    if PB_TOKEN:
        print("Using a supplied PocketBase authentication token; value hidden.")
    else:
        secret_or_prompt("PB_EMAIL", "PocketBase authorized email / identity: ", hidden=False)
        secret_or_prompt("PB_PASSWORD", "PocketBase authorized password: ", hidden=True)

    # Optional PB_URL may be set as a Colab Secret; otherwise use the URL in Ronan's source.
    if userdata is not None:
        try:
            pb_url_secret = userdata.get("PB_URL")
        except Exception:
            pb_url_secret = None
        if pb_url_secret:
            os.environ["PB_URL"] = pb_url_secret

    PB_URL = os.environ.get("PB_URL", "https://db.rcdonovan.com").rstrip("/")
    PB_AUTH_COLLECTION = "_superusers"  # confirmed by src/lib/server/pb.js and tools/sync-cfps.mjs
    print(f"Credentials loaded into runtime. API base: {PB_URL}; auth collection: {PB_AUTH_COLLECTION}.")
    print("Credential values and authentication tokens will not be displayed.")

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 3. Read-only PocketBase client: authenticate and fetch paginated collection records.
    # No create/update/delete endpoints are called by this class.
    import time
    import requests
    from urllib.parse import quote

    class PocketBaseReadOnlyClient:
        def __init__(self, base_url, identity=None, password=None, auth_collection="_superusers", timeout=45, token=None):
            self.base_url = base_url.rstrip("/")
            self.identity = identity
            self.password = password
            self.auth_collection = auth_collection
            self.timeout = timeout
            self.session = requests.Session()
            self.token = token
            if token:
                self.session.headers.update({"Authorization": f"Bearer {token}"})

        def authenticate(self):
            if self.token and not self.identity:
                return {}  # externally supplied PocketBase bearer token
            url = f"{self.base_url}/api/collections/{quote(self.auth_collection, safe='')}/auth-with-password"
            response = self.session.post(
                url,
                json={"identity": self.identity, "password": self.password},
                timeout=self.timeout,
            )
            if not response.ok:
                # Avoid echoing credentials/token; include only status and a shortened server message.
                try:
                    detail = response.json().get("message", "")
                except Exception:
                    detail = ""
                raise RuntimeError(
                    f"PocketBase authentication failed (HTTP {response.status_code}). "
                    f"Server message: {str(detail)[:300] or 'no message returned'}. "
                    "Confirm these are credentials authorized for _superusers, not merely Keycloak web-login credentials."
                )
            payload = response.json()
            if not payload.get("token"):
                raise RuntimeError("PocketBase authentication response did not contain a token.")
            self.token = payload["token"]
            self.session.headers.update({"Authorization": f"Bearer {self.token}"})
            return payload.get("record", {})

        def _get(self, path, params=None, retry_auth=True):
            if not self.token:
                self.authenticate()
            response = self.session.get(f"{self.base_url}{path}", params=params, timeout=self.timeout)
            if response.status_code == 401 and retry_auth:
                self.authenticate()
                response = self.session.get(f"{self.base_url}{path}", params=params, timeout=self.timeout)
            if not response.ok:
                try:
                    body = response.json()
                    message = body.get("message", "")
                except Exception:
                    message = ""
                raise RuntimeError(
                    f"PocketBase GET failed (HTTP {response.status_code}) at {path}. "
                    f"Server message: {str(message)[:300] or 'no message returned'}."
                )
            return response.json()

        def get_record(self, collection, record_id):
            return self._get(
                f"/api/collections/{quote(collection, safe='')}/records/{quote(record_id, safe='')}"
            )

        def list_records(self, collection, per_page=200, sort="-created", max_records=None):
            per_page = max(1, min(int(per_page), 500))
            page = 1
            rows = []
            advertised_total = None
            while True:
                params = {"page": page, "perPage": per_page}
                if sort:
                    params["sort"] = sort
                payload = self._get(
                    f"/api/collections/{quote(collection, safe='')}/records", params=params
                )
                items = payload.get("items", [])
                if advertised_total is None:
                    advertised_total = payload.get("totalItems")
                if not isinstance(items, list):
                    raise RuntimeError(f"Unexpected PocketBase list response for {collection!r}: items is not a list.")
                rows.extend(items)
                total_pages = int(payload.get("totalPages") or 0)
                if max_records is not None and len(rows) >= max_records:
                    rows = rows[:max_records]
                    break
                if not items or (total_pages and page >= total_pages):
                    break
                page += 1
                time.sleep(0.05)  # gentle pagination; does not materially slow small snapshots
            result = {
                "collection": collection,
                "records": rows,
                "advertised_total_items": advertised_total,
                "records_fetched": len(rows),
                "possibly_truncated": bool(max_records is not None and advertised_total is not None and len(rows) < advertised_total),
            }
            return result

    client = PocketBaseReadOnlyClient(
        PB_URL, os.environ.get("PB_EMAIL"), os.environ.get("PB_PASSWORD"), PB_AUTH_COLLECTION, token=PB_TOKEN
    )
    identity_record = client.authenticate()
    print("Authenticated successfully; token omitted.")
    print("Authenticated record ID:", identity_record.get("id", "not returned"))

## 4. Smoke-test only the source-confirmed collections

The notebook does not enumerate or dump every collection. It begins with the two collections identified in the application source:

- `cfps_designs`: the submitted CFPS formulations. The application stores recipe data in `design_json.reagent_list` and formulation metadata in `design_json.metadata`.
- `cfps_reagent_groups` / `t7a97pyfs8lx67w`: the reagent catalogue record used by the designer.

The `cfps_designs` list is intentionally login-gated in the application. Use this extraction only under the database access and data-sharing permission Ronan gave you. A successful superuser login does not imply that raw records may be republished.

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 4. Probe one list item and the known reagent-catalogue record before full retrieval.
    # All requests below are GET after authentication.
    DESIGNS_COLLECTION = "cfps_designs"
    REAGENT_GROUPS_COLLECTION = "cfps_reagent_groups"
    REAGENT_GROUPS_RECORD_ID = "t7a97pyfs8lx67w"

    sample_page = client.list_records(DESIGNS_COLLECTION, per_page=1, max_records=1)
    reagent_group_probe = client.get_record(REAGENT_GROUPS_COLLECTION, REAGENT_GROUPS_RECORD_ID)

    print("cfps_designs sample returned:", sample_page["records_fetched"])
    print("cfps_designs advertised total (if available):", sample_page["advertised_total_items"])
    print("Sample record keys:", sorted(sample_page["records"][0].keys()) if sample_page["records"] else "no rows visible")
    print("Reagent-group record ID returned:", reagent_group_probe.get("id", "missing id"))
    print("Reagent-group record keys:", sorted(reagent_group_probe.keys()))

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 5. Fetch a read-only snapshot and store raw responses outside the cloned repository.
    # Change MAX_DESIGNS only for a deliberate pilot; set it to None for all accessible records.
    from datetime import datetime, timezone
    import json

    MAX_DESIGNS = None  # e.g. 25 for a small test; None fetches all pages returned to this account
    SNAPSHOT_TIME = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    PRIVATE_RUN_DIR = PRIVATE_ROOT / SNAPSHOT_TIME
    PRIVATE_RUN_DIR.mkdir(parents=True, exist_ok=False)

    designs_result = client.list_records(
        DESIGNS_COLLECTION, per_page=200, sort="-created", max_records=MAX_DESIGNS
    )
    reagent_groups_record = client.get_record(REAGENT_GROUPS_COLLECTION, REAGENT_GROUPS_RECORD_ID)

    snapshot = {
        "metadata": {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "source_repository": REPO_URL,
            "source_repository_commit": REPO_SHA,
            "pocketbase_base_url": PB_URL,
            "authentication_collection": PB_AUTH_COLLECTION,
            "read_only_snapshot": True,
            "collections_requested": [DESIGNS_COLLECTION, REAGENT_GROUPS_COLLECTION],
            "designs_advertised_total_items": designs_result["advertised_total_items"],
            "designs_records_fetched": designs_result["records_fetched"],
            "designs_possibly_truncated": designs_result["possibly_truncated"],
            "note": "Raw records are private working data; do not commit or publish without coordinator approval."
        },
        "collections": {
            DESIGNS_COLLECTION: designs_result["records"],
            REAGENT_GROUPS_COLLECTION: [reagent_groups_record]
        }
    }
    RAW_SNAPSHOT_PATH = PRIVATE_RUN_DIR / "pocketbase_snapshot.json"
    RAW_SNAPSHOT_PATH.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Fetched {designs_result['records_fetched']} design records.")
    print(f"PocketBase advertised total: {designs_result['advertised_total_items']}.")
    print(f"Possibly truncated: {designs_result['possibly_truncated']}.")
    print(f"Raw snapshot stored outside Git at: {RAW_SNAPSHOT_PATH}")

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 6. Parse the source-defined data shape without changing the raw snapshot.
    import pandas as pd
    import numpy as np
    import hashlib


    def parse_json_value(value):
        if isinstance(value, str):
            stripped = value.strip()
            if not stripped:
                return None
            try:
                return json.loads(stripped)
            except Exception:
                return value
        return value


    def as_dict(value):
        value = parse_json_value(value)
        return value if isinstance(value, dict) else {}


    def norm_target(value):
        if not value:
            return "Unknown"
        s = str(value).strip()
        key = s.casefold()
        known = {"sfgfp": "sfGFP", "sf gfp": "sfGFP", "petase": "PETase", "reteplase": "Reteplase"}
        return known.get(key, s)


    def extract_design_row(rec):
        design = as_dict(rec.get("design_json"))
        metadata = as_dict(design.get("metadata"))
        reagent_list = as_dict(design.get("reagent_list"))
        sample_id = design.get("sample_id") or rec.get("title") or rec.get("id")
        batch_id = metadata.get("batch_id") or None
        return {
            "record_id": rec.get("id"),
            "sample_id": str(sample_id) if sample_id is not None else None,
            "created": rec.get("created"),
            "updated": rec.get("updated"),
            "protein_target": norm_target(metadata.get("protein_target")),
            "sample_type": design.get("sample_type"),
            "batch_id": str(batch_id) if batch_id else None,
            "batch_size": pd.to_numeric(metadata.get("batch_size"), errors="coerce"),
            "batch_index": pd.to_numeric(metadata.get("batch_index"), errors="coerce"),
            "has_experiment_hypothesis": bool(str(metadata.get("experiment_hypothesis") or "").strip()),
            "has_comments": bool(str(metadata.get("comments") or rec.get("rationale") or "").strip()),
            "has_reagent_list": bool(reagent_list),
            "reagent_list": reagent_list,
            # Keep text fields in memory for internal analysis, but never export them to reports.
            "experiment_hypothesis_private": str(metadata.get("experiment_hypothesis") or ""),
            "comments_private": str(metadata.get("comments") or rec.get("rationale") or ""),
            "reagent_list_json": json.dumps(reagent_list, sort_keys=True, separators=(",", ":")),
        }

    normalized_rows = [extract_design_row(rec) for rec in snapshot["collections"][DESIGNS_COLLECTION]]
    designs_df = pd.DataFrame(normalized_rows)
    if designs_df.empty:
        raise RuntimeError("No cfps_designs records were returned. Check access and the source schema before continuing.")

    def explode_reagents(df):
        out = []
        for row in df.to_dict(orient="records"):
            rlist = row.get("reagent_list") if isinstance(row.get("reagent_list"), dict) else {}
            for reagent_id, raw_volume in rlist.items():
                value = pd.to_numeric(raw_volume, errors="coerce")
                out.append({
                    "record_id": row.get("record_id"),
                    "sample_id": row.get("sample_id"),
                    "protein_target": row.get("protein_target"),
                    "batch_id": row.get("batch_id"),
                    "reagent_id": str(reagent_id),
                    "volume_nL_as_stored": float(value) if pd.notna(value) else np.nan,
                })
        return pd.DataFrame(out, columns=["record_id", "sample_id", "protein_target", "batch_id", "reagent_id", "volume_nL_as_stored"])

    reagent_volumes_df = explode_reagents(designs_df)

    def formulation_hash(reagent_list_json, protein_target):
        payload = f"{protein_target}|{reagent_list_json}".encode("utf-8")
        return hashlib.sha256(payload).hexdigest()

    designs_df["formulation_hash"] = designs_df.apply(
        lambda row: formulation_hash(row["reagent_list_json"], row["protein_target"]), axis=1
    )

    print(f"Normalized designs: {len(designs_df):,}")
    print(f"Rows with a usable reagent_list: {int(designs_df['has_reagent_list'].sum()):,}")
    print(f"Long-form reagent entries: {len(reagent_volumes_df):,}")
    print("Targets (including Unknown):")
    display(designs_df["protein_target"].value_counts(dropna=False).rename_axis("protein_target").to_frame("n_designs"))

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 7. Parse the source-confirmed reagent catalogue for labels and context (private working table).
    # The source normalizer supports a grouped `reagents` array; keep unknown shapes visible.
    reagent_record = snapshot["collections"][REAGENT_GROUPS_COLLECTION][0]

    def parse_catalog_groups(record):
        candidates = [
            record.get("reagentGroups"), record.get("reagent_groups"), record.get("groups"),
            record.get("data"), record.get("json"), record.get("payload"), record.get("content"),
            record.get("reagents")
        ]
        parsed_candidates = [parse_json_value(x) for x in candidates]
        for candidate in parsed_candidates:
            if isinstance(candidate, list):
                rows = []
                for group in candidate:
                    if not isinstance(group, dict):
                        continue
                    group_name = group.get("name") or group.get("group") or group.get("group_name") or "Uncategorized"
                    items = parse_json_value(group.get("reagents")) or parse_json_value(group.get("items")) or []
                    if isinstance(items, list):
                        for item in items:
                            if not isinstance(item, dict):
                                continue
                            rid = item.get("id") or item.get("reagent_id") or item.get("slug") or item.get("key")
                            if not rid:
                                continue
                            rows.append({
                                "group": group_name,
                                "reagent_id": str(rid),
                                "name": item.get("name") or item.get("reagent_name") or item.get("label") or str(rid),
                                "concentration_or_units": item.get("concentration") or item.get("units"),
                                "cost_per_ml": item.get("costPerMl", item.get("cost_per_ml", item.get("cost_per_mL"))),
                                "in_stock": item.get("in_stock", item.get("inStock")),
                                "hidden": item.get("hidden", False),
                            })
                if rows:
                    return rows
        return []

    catalog_rows = parse_catalog_groups(reagent_record)
    reagent_catalog_df = pd.DataFrame(catalog_rows)
    if reagent_catalog_df.empty:
        print("WARNING: No catalog rows recognized. Raw catalogue stays in the private snapshot; review the actual record shape before defining labels.")
    else:
        print(f"Parsed {len(reagent_catalog_df)} reagent-catalogue entries from the source record.")
        display(reagent_catalog_df.head(10))

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 8. Compute descriptive QC tables. These are aggregate/derived outputs suitable for review;
    # raw records, author fields, conversation links, comments, and row-level formulation exports
    # are deliberately not written into the repository reports directory.

    # Counts per target.
    counts_by_target = (
        designs_df.groupby("protein_target", dropna=False)
        .agg(
            n_designs=("record_id", "nunique"),
            n_with_reagent_list=("has_reagent_list", "sum"),
            n_with_hypothesis=("has_experiment_hypothesis", "sum"),
            n_with_comments=("has_comments", "sum"),
            n_with_batch_id=("batch_id", lambda s: s.notna().sum()),
        )
        .reset_index()
        .sort_values("protein_target")
    )

    # Date counts; only date and target counts are exported.
    designs_df["created_utc"] = pd.to_datetime(designs_df["created"], errors="coerce", utc=True)
    over_time = (
        designs_df.dropna(subset=["created_utc"])
        .assign(date=lambda x: x["created_utc"].dt.strftime("%Y-%m-%d"))
        .groupby(["date", "protein_target"], dropna=False)
        .size().rename("n_designs").reset_index()
        .sort_values(["date", "protein_target"])
    )

    # Formulation duplicate summary: repeated recipes are counted within a target.
    duplicate_groups = (
        designs_df.groupby(["protein_target", "formulation_hash"], dropna=False)
        .size().rename("n_records").reset_index()
    )
    duplicate_summary = (
        duplicate_groups.assign(is_duplicate=lambda x: x["n_records"] > 1)
        .groupby("protein_target", dropna=False)
        .agg(
            n_unique_formulations=("formulation_hash", "nunique"),
            n_duplicate_recipe_groups=("is_duplicate", "sum"),
            records_in_duplicate_groups=("n_records", lambda s: int(s[s > 1].sum())),
        )
        .reset_index()
    )

    # Reaction-total diagnostic. Repository source documents a 20,000 nL reaction as the current contract.
    # This is a warning table, not a filter: legacy designs may follow different assumptions.
    volume_totals = []
    for row in designs_df.to_dict(orient="records"):
        rlist = row.get("reagent_list") if isinstance(row.get("reagent_list"), dict) else {}
        numeric = [pd.to_numeric(v, errors="coerce") for v in rlist.values()]
        numeric = [float(v) for v in numeric if pd.notna(v)]
        total = sum(numeric) if numeric else np.nan
        volume_totals.append({
            "record_id": row.get("record_id"),
            "protein_target": row.get("protein_target"),
            "total_volume_nL": total,
            "matches_20000_nL": bool(pd.notna(total) and abs(total - 20000) < 1e-6),
            "has_numeric_volumes": bool(numeric),
        })
    volume_qc_df = pd.DataFrame(volume_totals)
    volume_qc_summary = (
        volume_qc_df.groupby("protein_target", dropna=False)
        .agg(
            n_designs=("record_id", "nunique"),
            n_with_numeric_volumes=("has_numeric_volumes", "sum"),
            n_matching_20000_nL=("matches_20000_nL", "sum"),
            median_total_volume_nL=("total_volume_nL", "median"),
        ).reset_index()
    )

    print("Counts by target")
    display(counts_by_target)
    print("Duplicate-formulation summary")
    display(duplicate_summary)
    print("20,000 nL QC diagnostic (mismatches are not automatically invalid)")
    display(volume_qc_summary)

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 9. Prepare a coverage matrix: percent of designs per target with each ingredient > 0 nL.
    # Missing fields are treated as absent for this descriptive chart, matching the app's sparse reagent_list shape.
    # Volumes remain `volume_nL_as_stored`; no concentration conversion is attempted.

    coverage_rows = []
    if not reagent_volumes_df.empty:
        valid_long = reagent_volumes_df.dropna(subset=["volume_nL_as_stored"]).copy()
        positive = valid_long[valid_long["volume_nL_as_stored"] > 0]
        n_per_target = designs_df.groupby("protein_target")["record_id"].nunique().to_dict()
        reagent_counts = positive.groupby(["protein_target", "reagent_id"])["record_id"].nunique().reset_index(name="n_designs_with_reagent")
        for r in reagent_counts.to_dict(orient="records"):
            denom = int(n_per_target.get(r["protein_target"], 0))
            coverage_rows.append({
                "protein_target": r["protein_target"],
                "reagent_id": r["reagent_id"],
                "n_designs_with_reagent": int(r["n_designs_with_reagent"]),
                "n_designs_in_target": denom,
                "prevalence_pct": (100 * r["n_designs_with_reagent"] / denom) if denom else np.nan,
            })
    coverage_df = pd.DataFrame(coverage_rows)
    if coverage_df.empty:
        print("No numeric reagent-volume data available for coverage analysis.")
    else:
        display(coverage_df.sort_values(["protein_target", "prevalence_pct"], ascending=[True, False]).head(20))

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 10. Create publication-friendly exploratory figures using aggregate data only.
    # The figure directory is in the repository; the private data tables are not.
    import matplotlib.pyplot as plt

    FIGURE_DIR = REPORT_DIR / "figures"
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    # A) Count of formulation records per target.
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    plot_counts = counts_by_target.sort_values("n_designs", ascending=False)
    ax.bar(plot_counts["protein_target"].astype(str), plot_counts["n_designs"])
    ax.set_title("CFPS formulation records by target")
    ax.set_xlabel("Target protein")
    ax.set_ylabel("Number of records")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "submissions_by_target.png", dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # B) Daily submission counts, if creation timestamps are available.
    if not over_time.empty:
        date_pivot = over_time.pivot(index="date", columns="protein_target", values="n_designs").fillna(0).sort_index()
        fig, ax = plt.subplots(figsize=(10, 5))
        for col in date_pivot.columns:
            ax.plot(date_pivot.index, date_pivot[col], marker="o", linewidth=1.5, label=str(col))
        ax.set_title("CFPS formulation records over time")
        ax.set_xlabel("Creation date (UTC)")
        ax.set_ylabel("Records per day")
        ax.tick_params(axis="x", rotation=45)
        ax.legend(title="Target", frameon=False)
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "submissions_over_time.png", dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    else:
        print("Skipped timeline figure: no valid created timestamps were found.")

    # C) Ingredient prevalence by target; show the 20 most commonly used ingredients overall.
    if not coverage_df.empty:
        top_reagents = (
            coverage_df.groupby("reagent_id")["n_designs_with_reagent"].sum()
            .sort_values(ascending=False).head(20).index.tolist()
        )
        cov = coverage_df[coverage_df["reagent_id"].isin(top_reagents)].pivot(
            index="reagent_id", columns="protein_target", values="prevalence_pct"
        )
        cov = cov.reindex(top_reagents)
        fig, ax = plt.subplots(figsize=(10, max(5, 0.34 * len(cov))))
        image = ax.imshow(cov.to_numpy(dtype=float), aspect="auto", interpolation="nearest", vmin=0, vmax=100)
        ax.set_title("Ingredient prevalence in formulations (top 20 ingredients)")
        ax.set_xlabel("Target protein")
        ax.set_ylabel("Reagent ID")
        ax.set_xticks(range(len(cov.columns)), labels=[str(c) for c in cov.columns], rotation=20, ha="right")
        ax.set_yticks(range(len(cov.index)), labels=[str(x) for x in cov.index])
        colorbar = fig.colorbar(image, ax=ax)
        colorbar.set_label("Designs containing reagent (%)")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "reagent_prevalence_by_target.png", dpi=220, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    else:
        print("Skipped reagent prevalence figure: no numeric reagent volumes were recognized.")

    print(f"Aggregate figures saved under {FIGURE_DIR}")

In [ ]:
# Optional section (enable RUN_POCKETBASE_FETCH in configuration cell).
if RUN_POCKETBASE_FETCH:
    # 11. Save aggregate reports inside the repo and the detailed, row-level working tables outside it.
    # The public/reviewable reports deliberately exclude author, author_sub, comments, hypotheses,
    # conversation links, sample IDs, and row-level recipe values.

    # Aggregate tables (appropriate for coordinator review; still review before publishing).
    counts_by_target.to_csv(REPORT_DIR / "counts_by_target.csv", index=False)
    duplicate_summary.to_csv(REPORT_DIR / "duplicate_formulation_summary.csv", index=False)
    volume_qc_summary.to_csv(REPORT_DIR / "volume_total_qc_by_target.csv", index=False)
    if not over_time.empty:
        over_time.to_csv(REPORT_DIR / "daily_counts_by_target.csv", index=False)
    if not coverage_df.empty:
        coverage_df.to_csv(REPORT_DIR / "reagent_coverage_by_target.csv", index=False)

    # Detailed intermediate tables stay outside the repo for internal analysis only.
    designs_private = designs_df.drop(columns=["created_utc"], errors="ignore").copy()
    designs_private["reagent_list"] = designs_private["reagent_list"].apply(lambda x: json.dumps(x, ensure_ascii=False, sort_keys=True) if isinstance(x, dict) else "{}")
    designs_private.to_csv(PRIVATE_RUN_DIR / "designs_normalized_private.csv", index=False)
    reagent_volumes_df.to_csv(PRIVATE_RUN_DIR / "reagent_volumes_long_private.csv", index=False)
    if not reagent_catalog_df.empty:
        reagent_catalog_df.to_csv(PRIVATE_RUN_DIR / "reagent_catalog_private.csv", index=False)

    manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_repository": REPO_URL,
        "source_repository_commit": REPO_SHA,
        "pocketbase_host": PB_URL,
        "auth_collection": PB_AUTH_COLLECTION,
        "collections": {
            DESIGNS_COLLECTION: {
                "fetched": designs_result["records_fetched"],
                "advertised_total_items": designs_result["advertised_total_items"],
                "possibly_truncated": designs_result["possibly_truncated"],
            },
            REAGENT_GROUPS_COLLECTION: {"record_id": REAGENT_GROUPS_RECORD_ID, "fetched": 1},
        },
        "derived_outputs": [
            "counts_by_target.csv",
            "duplicate_formulation_summary.csv",
            "volume_total_qc_by_target.csv",
            "daily_counts_by_target.csv" if not over_time.empty else None,
            "reagent_coverage_by_target.csv" if not coverage_df.empty else None,
            "figures/submissions_by_target.png",
            "figures/submissions_over_time.png" if not over_time.empty else None,
            "figures/reagent_prevalence_by_target.png" if not coverage_df.empty else None,
        ],
        "raw_data_stored_outside_repository": True,
        "publication_caveat": "Aggregate outputs only; coordinator approval and manual privacy review required before publication."
    }
    (REPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print("Aggregate/review outputs:", REPORT_DIR)
    print("Private detailed tables:", PRIVATE_RUN_DIR)
    print("Reports saved:")
    for p in sorted(REPORT_DIR.rglob("*")):
        if p.is_file():
            print(" -", p.relative_to(REPO_ROOT))

## 12. Recipe-vector pairwise correlation (independent of the published GP-informed model links)

Define each submitted recipe as a reagent-feature vector \(x_i\). The Spearman/Pearson
similarity \(\mathrm{corr}(x_i,x_j)\) correlates their patterns **across reagent features**, not
their observed yields and not the covariance of the published GP-informed Gaussian-process predictions.
A dissimilarity \(d_{ij}=1-r_{ij}\) with average-linkage clustering is only a
descriptive, model-free alternative. Features with zero variation within each protein are
dropped. Spearman on sparse recipes can be dominated by tied zeros and is a sensitivity
comparison—not the recommended chemical-space optimization distance.

The entire correlation class source is defined in the next cell, with no file writing.

In [ ]:
"""Reusable correlation-based clustering for entity-by-feature data.

The abstract class implements shared validation, pairwise correlation, conversion to
correlation dissimilarity, hierarchical clustering, output tables, and plots. A
subclass/adapter is responsible for converting a project-specific source (e.g. CFPS
submission JSON) into a numeric feature matrix and aligned metadata.

Interpretation warning
----------------------
For CFPS designs, the rows are submissions and the columns are reagent features.
The correlation between two rows measures similarity of their across-reagent *patterns*.
It does NOT measure correlation between experimental outcomes, causal relationships,
or expected information gain.
"""
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
import re
from typing import Any, Mapping, Optional

import numpy as np
import pandas as pd


def adjusted_rand_index(labels_a: Any, labels_b: Any) -> float:
    """Compute adjusted Rand index for two partitions over the same entities.

    Labels are aligned by position; callers must align entity IDs first. The score
    is invariant to arbitrary cluster label names and is intended for sensitivity
    analysis, not for choosing a preferred clustering by itself.
    """
    a = np.asarray(labels_a, dtype=object).ravel()
    b = np.asarray(labels_b, dtype=object).ravel()
    if len(a) != len(b):
        raise ValueError("Both label arrays must have the same length")
    n = len(a)
    if n < 2:
        return float("nan")
    contingency = pd.crosstab(pd.Series(a, name="a"), pd.Series(b, name="b")).to_numpy(dtype=float)
    choose2 = lambda x: x * (x - 1.0) / 2.0
    sum_cells = float(choose2(contingency).sum())
    sum_rows = float(choose2(contingency.sum(axis=1)).sum())
    sum_cols = float(choose2(contingency.sum(axis=0)).sum())
    total_pairs = float(choose2(n))
    expected = (sum_rows * sum_cols) / total_pairs
    maximum = 0.5 * (sum_rows + sum_cols)
    denominator = maximum - expected
    if abs(denominator) < np.finfo(float).eps:
        # Degenerate partitions: score identical pairwise memberships as 1, else 0.
        same = all((a[i] == a[j]) == (b[i] == b[j]) for i in range(n) for j in range(i + 1, n))
        return 1.0 if same else 0.0
    return float((sum_cells - expected) / denominator)


@dataclass
class CorrelationClusterResult:
    """Results for a single group of entities (e.g. one protein target)."""

    group_name: str
    feature_matrix: pd.DataFrame
    correlation_matrix: pd.DataFrame
    distance_matrix: pd.DataFrame
    labels: pd.Series
    linkage_matrix: np.ndarray
    leaf_order: list[str]
    dropped_entities: list[str]
    dropped_features: list[str]
    correlation_method: str
    linkage_method: str
    feature_mode: str


class AbstractPairwiseCorrelationClusterer(ABC):
    """Reusable analysis workflow for correlation-based hierarchical clustering.

    Subclasses must implement ``build_feature_matrix`` and ``build_metadata``.
    Feature matrix rows are entities and columns are numeric features. Entity indexes
    must be unique strings and must align with the metadata index. Missing feature
    values are rejected in the abstract layer: each adapter must state how its source
    encodes absence/missingness rather than having zeros guessed silently.

    ``1 - correlation`` is used as a dissimilarity. Since it is not guaranteed to be
    Euclidean, Ward/centroid/median linkage are intentionally disallowed; average or
    complete linkage are reasonable defaults for this distance.
    """

    ALLOWED_CORRELATIONS = {"pearson", "spearman"}
    ALLOWED_LINKAGES = {"average", "complete", "single", "weighted"}

    def __init__(
        self,
        *,
        correlation_method: str = "spearman",
        linkage_method: str = "average",
        n_clusters: int = 6,
        min_features: int = 3,
        group_by: Optional[str] = "protein_target",
    ) -> None:
        self.correlation_method = correlation_method.lower()
        self.linkage_method = linkage_method.lower()
        self.n_clusters = int(n_clusters)
        self.min_features = int(min_features)
        self.group_by = group_by

        if self.correlation_method not in self.ALLOWED_CORRELATIONS:
            raise ValueError(f"correlation_method must be one of {sorted(self.ALLOWED_CORRELATIONS)}")
        if self.linkage_method not in self.ALLOWED_LINKAGES:
            raise ValueError(
                f"linkage_method must be one of {sorted(self.ALLOWED_LINKAGES)}; "
                "Ward/centroid/median are not supported for 1-r dissimilarity."
            )
        if self.n_clusters < 1:
            raise ValueError("n_clusters must be >= 1")
        if self.min_features < 2:
            raise ValueError("min_features must be >= 2")

        self.results_: dict[str, CorrelationClusterResult] = {}
        self.skipped_groups_: dict[str, str] = {}
        self.feature_matrix_: Optional[pd.DataFrame] = None
        self.metadata_: Optional[pd.DataFrame] = None

    @abstractmethod
    def build_feature_matrix(self) -> pd.DataFrame:
        """Return a numeric DataFrame: entities as rows, features as columns."""
        raise NotImplementedError

    @abstractmethod
    def build_metadata(self) -> pd.DataFrame:
        """Return entity metadata indexed by the same entity IDs as the feature matrix."""
        raise NotImplementedError

    @staticmethod
    def _safe_group_name(value: Any) -> str:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return "Unknown"
        text = str(value).strip()
        return text if text else "Unknown"

    @staticmethod
    def _slug(value: str) -> str:
        slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_.-")
        return slug or "group"

    def validate_inputs(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Build and validate matrices; does not mutate the adapter's source records."""
        X = self.build_feature_matrix().copy()
        metadata = self.build_metadata().copy()
        if not isinstance(X, pd.DataFrame) or X.empty:
            raise ValueError("build_feature_matrix() must return a non-empty pandas DataFrame")
        if not isinstance(metadata, pd.DataFrame):
            raise TypeError("build_metadata() must return a pandas DataFrame")

        X.index = X.index.map(str)
        metadata.index = metadata.index.map(str)
        if X.index.has_duplicates:
            raise ValueError("Feature matrix index must be unique; create stable unique entity IDs in the adapter")
        if metadata.index.has_duplicates:
            raise ValueError("Metadata index must be unique")
        if not X.index.equals(metadata.index):
            missing = sorted(set(X.index) - set(metadata.index))[:5]
            extra = sorted(set(metadata.index) - set(X.index))[:5]
            raise ValueError(f"Feature/metadata indexes must match and be ordered alike. Missing metadata={missing}; extra metadata={extra}")

        X = X.apply(pd.to_numeric, errors="coerce")
        if X.columns.duplicated().any():
            raise ValueError("Feature names must be unique")
        if X.isna().any().any():
            n = int(X.isna().sum().sum())
            raise ValueError(
                f"Feature matrix contains {n} missing/non-numeric values. Resolve missingness explicitly in the adapter; "
                "do not silently turn unknown values into zero."
            )
        values = X.to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError("Feature matrix contains infinite values")
        self.feature_matrix_ = X
        self.metadata_ = metadata
        return X, metadata

    def _fit_one_group(self, group_name: str, X: pd.DataFrame) -> Optional[CorrelationClusterResult]:
        if len(X) < 2:
            self.skipped_groups_[group_name] = f"Only {len(X)} entity/entities; need at least 2."
            return None

        # Invariant features carry no discriminatory information within this group.
        variable_mask = X.nunique(axis=0, dropna=False) > 1
        dropped_features = X.columns[~variable_mask].astype(str).tolist()
        Xv = X.loc[:, variable_mask].copy()
        if Xv.shape[1] < self.min_features:
            self.skipped_groups_[group_name] = (
                f"Only {Xv.shape[1]} variable feature(s) after removing constants; "
                f"need at least {self.min_features}."
            )
            return None

        # Correlation is undefined for an entity vector with zero variance across features.
        row_sd = Xv.std(axis=1, ddof=0)
        valid_rows = row_sd > np.finfo(float).eps
        dropped_entities = Xv.index[~valid_rows].astype(str).tolist()
        Xv = Xv.loc[valid_rows]
        if len(Xv) < 2:
            self.skipped_groups_[group_name] = "Fewer than two non-constant entity vectors remain."
            return None

        correlation = Xv.T.corr(method=self.correlation_method, min_periods=self.min_features)
        correlation = correlation.clip(lower=-1.0, upper=1.0)
        correlation = (correlation + correlation.T) / 2.0
        np.fill_diagonal(correlation.values, 1.0)
        if not np.isfinite(correlation.to_numpy(dtype=float)).all():
            raise ValueError(
                f"Group {group_name!r} produced undefined pairwise correlations. "
                "Check sparse/tied features or raise feature quality before clustering."
            )

        distance = (1.0 - correlation).clip(lower=0.0, upper=2.0)
        distance = (distance + distance.T) / 2.0
        np.fill_diagonal(distance.values, 0.0)

        from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
        from scipy.spatial.distance import squareform

        condensed = squareform(distance.to_numpy(dtype=float), checks=True)
        Z = linkage(condensed, method=self.linkage_method, optimal_ordering=True)
        k = min(self.n_clusters, len(Xv))
        cluster_numbers = fcluster(Z, t=k, criterion="maxclust")
        labels = pd.Series(
            [f"C{int(v):02d}" for v in cluster_numbers], index=Xv.index, name="cluster_id"
        )
        order = leaves_list(Z)
        leaf_order = Xv.index[order].astype(str).tolist()

        return CorrelationClusterResult(
            group_name=group_name,
            feature_matrix=Xv,
            correlation_matrix=correlation,
            distance_matrix=distance,
            labels=labels,
            linkage_matrix=Z,
            leaf_order=leaf_order,
            dropped_entities=dropped_entities,
            dropped_features=dropped_features,
            correlation_method=self.correlation_method,
            linkage_method=self.linkage_method,
            feature_mode=getattr(self, "feature_mode", "adapter-defined"),
        )

    def fit(self) -> "AbstractPairwiseCorrelationClusterer":
        """Run one clustering per group (protein target by default)."""
        X, metadata = self.validate_inputs()
        self.results_ = {}
        self.skipped_groups_ = {}

        if self.group_by and self.group_by in metadata.columns:
            groups = metadata[self.group_by].map(self._safe_group_name)
            grouped_indices = [(name, metadata.index[groups == name]) for name in pd.unique(groups)]
        elif self.group_by is None:
            grouped_indices = [("All", metadata.index)]
        else:
            raise ValueError(f"group_by={self.group_by!r} is not a metadata column; available: {list(metadata.columns)}")

        for name, ids in grouped_indices:
            result = self._fit_one_group(str(name), X.loc[ids])
            if result is not None:
                self.results_[str(name)] = result

        if not self.results_:
            raise ValueError(f"No groups could be clustered. Skip reasons: {self.skipped_groups_}")
        return self

    def labels_frame(self) -> pd.DataFrame:
        """Return row-level labels plus metadata. Treat as private unless approved for sharing."""
        if not self.results_:
            raise RuntimeError("Call fit() before requesting labels")
        assert self.metadata_ is not None
        frames = []
        for group_name, result in self.results_.items():
            part = self.metadata_.loc[result.labels.index].copy()
            part.insert(0, "entity_id", part.index.astype(str))
            part["cluster_group"] = group_name
            part["cluster_id"] = result.labels.astype(str)
            part["correlation_method"] = result.correlation_method
            part["linkage_method"] = result.linkage_method
            part["feature_mode"] = result.feature_mode
            frames.append(part.reset_index(drop=True))
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def compare_with(self, other: "AbstractPairwiseCorrelationClusterer") -> pd.DataFrame:
        """Compare this fit with another fit using adjusted Rand index by group.

        Only entities clustered by both runs are compared; the overlap count/fraction
        is reported so that an apparently good score based on a small subset is visible.
        """
        if not self.results_ or not other.results_:
            raise RuntimeError("Both clusterers must be fitted before compare_with()")
        rows = []
        for group in sorted(set(self.results_) & set(other.results_)):
            left = self.results_[group].labels.rename("left")
            right = other.results_[group].labels.rename("right")
            joined = pd.concat([left, right], axis=1, join="inner").dropna()
            n_left = len(left)
            n_right = len(right)
            rows.append({
                "group": group,
                "n_overlap": int(len(joined)),
                "n_left": int(n_left),
                "n_right": int(n_right),
                "overlap_pct_of_left": (100.0 * len(joined) / n_left) if n_left else np.nan,
                "overlap_pct_of_right": (100.0 * len(joined) / n_right) if n_right else np.nan,
                "adjusted_rand_index": adjusted_rand_index(joined["left"], joined["right"]) if len(joined) >= 2 else np.nan,
            })
        return pd.DataFrame(rows, columns=["group", "n_overlap", "n_left", "n_right", "overlap_pct_of_left", "overlap_pct_of_right", "adjusted_rand_index"])

    def cluster_sizes_frame(self) -> pd.DataFrame:
        """Aggregate cluster sizes; review before publication because groups can be small."""
        rows = []
        for group_name, result in self.results_.items():
            counts = result.labels.value_counts().sort_index()
            for cluster_id, count in counts.items():
                rows.append({"group": group_name, "cluster_id": cluster_id, "n_entities": int(count)})
        return pd.DataFrame(rows, columns=["group", "cluster_id", "n_entities"])

    def save_private_outputs(self, output_dir: str | Path) -> list[Path]:
        """Write row-level matrices/labels to a caller-specified private directory."""
        if not self.results_:
            raise RuntimeError("Call fit() before saving outputs")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        written: list[Path] = []
        label_path = out / "cluster_labels_private.csv"
        self.labels_frame().to_csv(label_path, index=False)
        written.append(label_path)
        for name, result in self.results_.items():
            slug = self._slug(name)
            for filename, table in (
                (f"correlation_{slug}_private.csv", result.correlation_matrix),
                (f"distance_{slug}_private.csv", result.distance_matrix),
                (f"features_{slug}_private.csv", result.feature_matrix),
            ):
                path = out / filename
                table.to_csv(path, index=True)
                written.append(path)
            path = out / f"cluster_sizes_{slug}_aggregate.csv"
            result.labels.value_counts().rename_axis("cluster_id").rename("n_entities").reset_index().to_csv(path, index=False)
            written.append(path)
        return written

    def plot_correlation_heatmap(
        self,
        group_name: str,
        *,
        annotate_max_entities: int = 35,
        figsize: Optional[tuple[float, float]] = None,
        save_path: str | Path | None = None,
    ):
        """Plot pairwise correlations, ordered by hierarchical leaves and cluster."""
        import matplotlib.pyplot as plt

        result = self.results_[group_name]
        order = result.leaf_order
        corr = result.correlation_matrix.loc[order, order]
        n = len(order)
        if figsize is None:
            side = min(16.0, max(6.0, n * 0.035))
            figsize = (side, side)
        fig, ax = plt.subplots(figsize=figsize)
        image = ax.imshow(corr.to_numpy(dtype=float), vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest", aspect="auto")
        cb = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label(f"{result.correlation_method.title()} correlation (recipe vectors)")
        ax.set_title(f"Pairwise recipe correlation — {group_name}\n{n} submissions · {result.linkage_method} linkage · {result.feature_mode} features")
        if n <= annotate_max_entities:
            ax.set_xticks(range(n), labels=order, rotation=90, fontsize=7)
            ax.set_yticks(range(n), labels=order, fontsize=7)
        else:
            ax.set_xticks([])
            ax.set_yticks([])
        # Draw separators between adjacent clusters in leaf order.
        ordered_labels = result.labels.loc[order].astype(str).tolist()
        for i in range(1, n):
            if ordered_labels[i] != ordered_labels[i - 1]:
                ax.axhline(i - 0.5, color="black", linewidth=0.7, alpha=0.7)
                ax.axvline(i - 0.5, color="black", linewidth=0.7, alpha=0.7)
        fig.tight_layout()
        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=220, bbox_inches="tight")
        return fig, ax

    def plot_dendrogram(
        self,
        group_name: str,
        *,
        show_labels_max_entities: int = 60,
        figsize: Optional[tuple[float, float]] = None,
        save_path: str | Path | None = None,
    ):
        """Plot a hierarchical dendrogram; suppress labels on large groups."""
        import matplotlib.pyplot as plt
        from scipy.cluster.hierarchy import dendrogram

        result = self.results_[group_name]
        n = len(result.labels)
        if figsize is None:
            figsize = (max(10.0, min(22.0, n * 0.06)), 6.5)
        fig, ax = plt.subplots(figsize=figsize)
        dendrogram(
            result.linkage_matrix,
            labels=result.feature_matrix.index.astype(str).tolist() if n <= show_labels_max_entities else None,
            leaf_rotation=90,
            leaf_font_size=7,
            ax=ax,
            no_labels=(n > show_labels_max_entities),
            color_threshold=None,
        )
        ax.set_title(f"Hierarchical clustering — {group_name}")
        ax.set_xlabel("Submission" if n <= show_labels_max_entities else "Submissions (labels suppressed)")
        ax.set_ylabel("1 − {0} correlation dissimilarity".format(result.correlation_method))
        fig.tight_layout()
        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=220, bbox_inches="tight")
        return fig, ax

    def plot_cluster_sizes(
        self,
        group_name: str,
        *,
        figsize: tuple[float, float] = (7.5, 4.5),
        save_path: str | Path | None = None,
    ):
        """Plot the number of entities in each cluster."""
        import matplotlib.pyplot as plt

        result = self.results_[group_name]
        counts = result.labels.value_counts().sort_index()
        fig, ax = plt.subplots(figsize=figsize)
        ax.bar(counts.index.astype(str), counts.values)
        ax.set_title(f"Cluster sizes — {group_name}")
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Number of submissions")
        fig.tight_layout()
        if save_path is not None:
            Path(save_path).parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=220, bbox_inches="tight")
        return fig, ax


class CFPSSubmissionCorrelationAdapter(AbstractPairwiseCorrelationClusterer):
    """Adapter from the CFPS coopetition snapshot's records to recipe feature vectors.

    Accepted input shapes:
      * raw ``cfps_designs`` record list;
      * snapshot mapping with ``collections.cfps_designs``;
      * iterable PocketBase records with ``design_json`` as a dict or JSON string;
      * a single submission payload (``{"sample": ...}``) or batch payload
        (``{"samples": [...]}``) matching the coopetition agent instructions.

    In the agent/API contract, a sample has ``sample_id``, ``sample_type``,
    ``reagent_list`` (reagent ID -> volume in nL), and ``metadata``; the live PocketBase
    collection wraps this under ``design_json``. Omitted reagent keys are treated as
    zero volume because the submission contract uses a sparse reagent mapping. A key
    present with a non-numeric value is not silently treated as zero; that row's feature
    value is marked invalid and input validation will fail loudly.

    ``feature_mode`` options:
      * ``volume``: recorded nL volumes (default);
      * ``log1p_volume``: log(1 + nL), to reduce domination by large volume values;
      * ``presence``: 1 if volume > 0, otherwise 0.
    """

    FEATURE_MODES = {"volume", "log1p_volume", "presence"}

    def __init__(
        self,
        source: Any,
        *,
        feature_mode: str = "volume",
        targets: Optional[list[str]] = None,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        if feature_mode not in self.FEATURE_MODES:
            raise ValueError(f"feature_mode must be one of {sorted(self.FEATURE_MODES)}")
        self.feature_mode = feature_mode
        self.targets = set(targets) if targets is not None else None
        self.records = self._extract_records(source)
        self._parsed_rows = self._parse_records(self.records)
        if self.targets is not None:
            allowed = {self._normalize_target(t) for t in self.targets}
            self._parsed_rows = [r for r in self._parsed_rows if r["protein_target"] in allowed]
        if not self._parsed_rows:
            raise ValueError("No CFPS submission records remain after parsing/target filtering")

    @staticmethod
    def _parse_json(value: Any) -> Any:
        if isinstance(value, str):
            text = value.strip()
            if not text:
                return None
            try:
                import json
                return json.loads(text)
            except (TypeError, ValueError):
                return value
        return value

    @classmethod
    def _as_dict(cls, value: Any) -> dict:
        parsed = cls._parse_json(value)
        return parsed if isinstance(parsed, dict) else {}

    @classmethod
    def _extract_records(cls, source: Any) -> list[dict]:
        inherited_batch_id = None
        if isinstance(source, Mapping):
            collections = source.get("collections", source)
            if isinstance(collections, Mapping) and "cfps_designs" in collections:
                records = collections["cfps_designs"]
            elif isinstance(source.get("samples"), (list, tuple)):
                # Agent API batch payload: {experiment_hypothesis, batch_id, samples: [...]}
                records = source["samples"]
                inherited_batch_id = source.get("batch_id") or source.get("batchId")
            elif isinstance(source.get("sample"), Mapping):
                # Agent API single-sample payload: {sample: {...}}
                records = [source]
            elif isinstance(source.get("reagent_list"), Mapping):
                # Bare sample object: {sample_id, sample_type, reagent_list, metadata}
                records = [source]
            else:
                raise ValueError(
                    "Mapping input must be a snapshot containing collections['cfps_designs'], "
                    "an agent payload with 'sample'/'samples', or a bare sample object."
                )
        else:
            records = source
        if not isinstance(records, (list, tuple)):
            raise TypeError("source must be a list of records, a snapshot mapping, or an agent sample payload")
        output = []
        for rec in records:
            if not isinstance(rec, Mapping):
                continue
            item = dict(rec)
            if inherited_batch_id is not None:
                item.setdefault("batch_id", inherited_batch_id)
            output.append(item)
        return output

    @staticmethod
    def _normalize_target(value: Any) -> str:
        if value is None or not str(value).strip():
            return "Unknown"
        raw = str(value).strip()
        known = {"sfgfp": "sfGFP", "sf gfp": "sfGFP", "petase": "PETase", "reteplase": "Reteplase"}
        return known.get(raw.casefold(), raw)

    def _parse_records(self, records: list[dict]) -> list[dict]:
        parsed_rows = []
        seen_entity_ids: set[str] = set()
        for idx, record in enumerate(records):
            design = self._as_dict(record.get("design_json"))
            # Agent POST payloads wrap a single composition under `sample`; other exports
            # may keep its fields directly on the record.
            if not design and isinstance(record.get("sample"), Mapping):
                design = self._as_dict(record.get("sample"))
            if not design and isinstance(record.get("reagent_list"), Mapping):
                design = record
            metadata = self._as_dict(design.get("metadata"))
            raw_reagents = self._as_dict(design.get("reagent_list"))
            target_value = metadata.get("protein_target", design.get("protein_target", record.get("protein_target", record.get("proteinTarget"))))
            target = self._normalize_target(target_value)
            if not raw_reagents:
                continue
            base_id = record.get("id") or design.get("sample_id") or record.get("title") or f"row_{idx:06d}"
            entity_id = str(base_id)
            if entity_id in seen_entity_ids:
                entity_id = f"{entity_id}__row_{idx:06d}"
            seen_entity_ids.add(entity_id)

            values: dict[str, float] = {}
            bad_fields: list[str] = []
            for reagent_id, raw_value in raw_reagents.items():
                reagent = str(reagent_id)
                try:
                    value = float(raw_value)
                except (TypeError, ValueError):
                    bad_fields.append(reagent)
                    continue
                if not np.isfinite(value) or value < 0:
                    bad_fields.append(reagent)
                    continue
                values[reagent] = value
            if bad_fields:
                raise ValueError(
                    f"Record {entity_id!r} has invalid/nonnegative-volume violations in reagent field(s): {bad_fields[:8]}"
                )
            if not values:
                continue
            if self.feature_mode == "presence":
                values = {k: float(v > 0) for k, v in values.items()}
            elif self.feature_mode == "log1p_volume":
                values = {k: float(np.log1p(v)) for k, v in values.items()}

            parsed_rows.append({
                "entity_id": entity_id,
                "record_id": str(record.get("id") or entity_id),
                "sample_id": design.get("sample_id") or record.get("title"),
                "protein_target": target,
                "batch_id": metadata.get("batch_id") or record.get("batch_id") or record.get("batchId"),
                "sample_type": design.get("sample_type"),
                "created": record.get("created"),
                "reagents": values,
            })
        return parsed_rows

    def build_feature_matrix(self) -> pd.DataFrame:
        # Missing mapping keys represent unadded reagents and are therefore zero in this API contract.
        feature_names = sorted({name for row in self._parsed_rows for name in row["reagents"]})
        rows = []
        ids = []
        for row in self._parsed_rows:
            rows.append({name: row["reagents"].get(name, 0.0) for name in feature_names})
            ids.append(row["entity_id"])
        return pd.DataFrame(rows, index=pd.Index(ids, name="entity_id"), columns=feature_names, dtype=float)

    def build_metadata(self) -> pd.DataFrame:
        columns = ["record_id", "sample_id", "protein_target", "batch_id", "sample_type", "created"]
        rows = []
        ids = []
        for row in self._parsed_rows:
            ids.append(row["entity_id"])
            rows.append({key: row.get(key) for key in columns})
        return pd.DataFrame(rows, index=pd.Index(ids, name="entity_id"), columns=columns)

    def feature_prevalence(self) -> pd.DataFrame:
        """Return a private QC table showing how frequently each reagent is present."""
        X = self.build_feature_matrix()
        return pd.DataFrame({
            "reagent_id": X.columns.astype(str),
            "n_entities_present": (X > 0).sum(axis=0).to_numpy(dtype=int),
            "n_entities": len(X),
            "prevalence_pct": 100.0 * (X > 0).mean(axis=0).to_numpy(dtype=float),
        }).sort_values(["prevalence_pct", "reagent_id"], ascending=[False, True]).reset_index(drop=True)


In [ ]:
# Optional section (enable RUN_CORRELATION_CLUSTERING and RUN_POCKETBASE_FETCH in configuration cell).
if RUN_CORRELATION_CLUSTERING and RUN_POCKETBASE_FETCH:
    # 12.2 Fit the primary clusterer on the fetched snapshot.
    # Change these settings deliberately; do not compare cluster labels across runs without checking shared IDs.
    import sys
    import json
    from datetime import datetime, timezone
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

    # CFPSSubmissionCorrelationAdapter defined above (self-contained).

    CORRELATION_METHOD = "spearman"   # "spearman" or "pearson"
    FEATURE_MODE = "volume"           # "volume", "log1p_volume", or "presence"
    LINKAGE_METHOD = "average"        # average/complete/single/weighted; Ward is not valid for 1-r dissimilarity
    N_CLUSTERS_PER_TARGET = 6
    MIN_VARIABLE_FEATURES = 3

    clusterer = CFPSSubmissionCorrelationAdapter(
        snapshot,
        feature_mode=FEATURE_MODE,
        correlation_method=CORRELATION_METHOD,
        linkage_method=LINKAGE_METHOD,
        n_clusters=N_CLUSTERS_PER_TARGET,
        min_features=MIN_VARIABLE_FEATURES,
        group_by="protein_target",
    ).fit()

    print("Clustered groups:", list(clusterer.results_))
    print("Skipped groups:", clusterer.skipped_groups_ or "none")
    for group_name, result in clusterer.results_.items():
        print(
            f"{group_name}: n={len(result.labels)}, features retained={result.feature_matrix.shape[1]}, "
            f"constant features dropped={len(result.dropped_features)}, "
            f"constant/invalid recipe vectors dropped={len(result.dropped_entities)}, "
            f"clusters={result.labels.nunique()}"
        )

In [ ]:
# Optional section (enable RUN_CORRELATION_CLUSTERING and RUN_POCKETBASE_FETCH in configuration cell).
if RUN_CORRELATION_CLUSTERING and RUN_POCKETBASE_FETCH:
    # 12.3 Save row-level matrices privately and render three diagnostics per target.
    # These files contain row-level competition data/derived results. They are NOT written to REPORT_DIR.
    CLUSTER_PRIVATE_DIR = PRIVATE_RUN_DIR / "pairwise_correlation_clustering"
    cluster_files = clusterer.save_private_outputs(CLUSTER_PRIVATE_DIR)

    cluster_figure_dir = CLUSTER_PRIVATE_DIR / "figures"
    cluster_figure_dir.mkdir(parents=True, exist_ok=True)
    for group_name, result in clusterer.results_.items():
        safe_name = clusterer._slug(group_name)
        fig, ax = clusterer.plot_correlation_heatmap(
            group_name,
            save_path=cluster_figure_dir / f"correlation_heatmap_{safe_name}_private.png",
        )
        plt.show()
        plt.close(fig)

        fig, ax = clusterer.plot_dendrogram(
            group_name,
            save_path=cluster_figure_dir / f"dendrogram_{safe_name}_private.png",
        )
        plt.show()
        plt.close(fig)

        fig, ax = clusterer.plot_cluster_sizes(
            group_name,
            save_path=cluster_figure_dir / f"cluster_sizes_{safe_name}_private.png",
        )
        plt.show()
        plt.close(fig)

    clustering_manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "interpretation": "Pairwise correlations between submission recipe vectors across reagent features; not experimental-outcome correlations.",
        "source_collection": DESIGNS_COLLECTION,
        "source_repository_commit": REPO_SHA,
        "feature_mode": FEATURE_MODE,
        "correlation_method": CORRELATION_METHOD,
        "linkage_method": LINKAGE_METHOD,
        "requested_clusters_per_group": N_CLUSTERS_PER_TARGET,
        "min_variable_features": MIN_VARIABLE_FEATURES,
        "group_by": "protein_target",
        "groups": {
            name: {
                "n_entities_clustered": int(len(result.labels)),
                "n_features_retained": int(result.feature_matrix.shape[1]),
                "n_constant_features_dropped": int(len(result.dropped_features)),
                "n_constant_entity_vectors_dropped": int(len(result.dropped_entities)),
                "cluster_sizes": {str(k): int(v) for k, v in result.labels.value_counts().sort_index().items()},
            }
            for name, result in clusterer.results_.items()
        },
        "skipped_groups": clusterer.skipped_groups_,
        "private_row_level_outputs": True,
        "sharing_status": "Not approved for publication; inspect permissions and review outputs first.",
    }
    (CLUSTER_PRIVATE_DIR / "clustering_manifest_private.json").write_text(
        json.dumps(clustering_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Private clustering output directory: {CLUSTER_PRIVATE_DIR}")
    print("Wrote", len(cluster_files), "private CSVs plus the manifest and figures.")

In [ ]:
# Optional section (enable RUN_CORRELATION_CLUSTERING and RUN_POCKETBASE_FETCH in configuration cell).
if RUN_CORRELATION_CLUSTERING and RUN_POCKETBASE_FETCH:
    # 12.4 Sensitivity analysis: do clusters change under a different correlation/feature definition?
    # This uses ARI on entities clustered by both runs; overlap fractions are reported by compare_with().
    SENSITIVITY_CONFIGS = [
        ("pearson_volume", "pearson", "volume"),
        ("spearman_presence", "spearman", "presence"),
    ]
    sensitivity_clusterers = {"primary": clusterer}
    sensitivity_failures = {}
    for label, method, feature_mode in SENSITIVITY_CONFIGS:
        candidate = CFPSSubmissionCorrelationAdapter(
            snapshot,
            feature_mode=feature_mode,
            correlation_method=method,
            linkage_method=LINKAGE_METHOD,
            n_clusters=N_CLUSTERS_PER_TARGET,
            min_features=MIN_VARIABLE_FEATURES,
            group_by="protein_target",
        )
        try:
            candidate.fit()
            sensitivity_clusterers[label] = candidate
        except Exception as exc:
            sensitivity_failures[label] = f"{type(exc).__name__}: {str(exc)[:250]}"

    sensitivity_frames = []
    for label, candidate in sensitivity_clusterers.items():
        if label == "primary":
            continue
        comparison = clusterer.compare_with(candidate)
        if not comparison.empty:
            comparison.insert(0, "comparison", f"primary_vs_{label}")
            sensitivity_frames.append(comparison)

    if sensitivity_frames:
        sensitivity_df = pd.concat(sensitivity_frames, ignore_index=True)
        display(sensitivity_df)
        sensitivity_df.to_csv(CLUSTER_PRIVATE_DIR / "sensitivity_ari_private.csv", index=False)
    else:
        sensitivity_df = pd.DataFrame()
        print("No successful sensitivity comparisons were available.")

    (CLUSTER_PRIVATE_DIR / "sensitivity_failures_private.json").write_text(
        json.dumps(sensitivity_failures, indent=2), encoding="utf-8"
    )
    print("Sensitivity comparison files are private:", CLUSTER_PRIVATE_DIR)

## 13. GP-informed design audit: exact submitted recipes, no GP retraining

**Source:** the published GP-informed published `clusters_designs.json`. The notebook reads exactly this
file from `GP_DESIGNS_PATH` (upload it using Colab's left-side Files pane). The
compressed `clusters.json.gz` contains presentation data and rounded link matrices;
upload it optionally to obtain a separate link-matrix audit.

**Audit subroutine:** `audit_gp_model_snapshot(data, working_quota=...)`.

- Preserve stable original IDs, targets, roles, and pre-run forecasts.
- Verify nominal 20,000 nL volumes; identify duplicate dispensing formulations
  ignoring compensating water; count *unique* first/check/independent/sweep runs.
- For every cluster check, compare it against its group's first run and count all
  changed reagents (not merely the labeled change).
- Find existing exact matched counterparts and audit ≥3-level single-factor sweeps.
- Emit a manifest with SHA-256 of input, package versions, parameters, and private CSVs.

**Caveat:** dispensing volumes are not automatically concentrations. No inferred stock
molarities. These are pre-experimental selection diagnostics, not measured effects.

In [ ]:
"""Reproducible, pre-experimental CFPS candidate audit and portfolio diagnostics.

Inputs: the dated GP-informed clusters_designs.json export (no measurements required).
All output is a description of submitted recipes or PRE-RUN model predictions.
No code here retrains the published GP or estimates empirical response correlation.

The same source is embedded *verbatim* into the independently executable Colab notebook.
Version: 1.1.0 (2026-10-27).
"""
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import gzip
import json
import math
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

VERSION = "1.1.0"
WORKING_QUOTA = 320  # NOT verified final competition rules
REACTION_NL = 20_000.0
EXCLUDED_BALANCERS = frozenset({"nuclease_free_water"})
BASELINE_ROLES = frozenset({"first run", "check run", "independent run", "sweep run"})
TARGETS = ("sfGFP", "PETase", "Reteplase")


def canonical_recipe(recipe: Mapping[str, Any], *, exclude_water: bool = True) -> tuple:
    """Immutable recipe identity; zeros and absent keys are equivalent.

    This identifies exact *dispensing recipes*, NOT chemical equivalence when stock
    concentrations or identities differ. Does not compare across protein targets.
    """
    if not isinstance(recipe, Mapping):
        raise TypeError("recipe must be a mapping")
    out = []
    for key, raw in recipe.items():
        if exclude_water and key in EXCLUDED_BALANCERS:
            continue
        if isinstance(raw, bool):
            raise ValueError(f"Boolean volume for {key}")
        try:
            vol = float(raw)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Non-numeric volume for {key}: {raw!r}") from exc
        if not np.isfinite(vol) or vol < 0:
            raise ValueError(f"Invalid volume for {key}: {vol!r}")
        if vol != 0:
            out.append((str(key), round(vol, 9)))
    return tuple(sorted(out))


def signature_except(recipe: Mapping[str, Any], factor: str) -> tuple:
    return tuple((k, v) for k, v in canonical_recipe(recipe) if k != factor)


def recipe_factor_value(recipe: Mapping[str, Any], factor: str) -> float:
    return float(recipe.get(factor, 0))


def changed_reagents(recipe_a: Mapping[str, Any], recipe_b: Mapping[str, Any]) -> list[str]:
    a, b = dict(canonical_recipe(recipe_a)), dict(canonical_recipe(recipe_b))
    return [key for key in sorted(a.keys() | b.keys()) if not np.isclose(a.get(key, 0), b.get(key, 0), rtol=0, atol=1e-8)]


def input_sha256(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _factor_key_from_display(ingredient: str, record: Mapping[str, Any]) -> str:
    """Resolve check's display label to stored reagent key using check's own data.

    Prefer exact `group.checks.ingredient` mapped through displayed names from
    vsGinkgoPlates or sweeps; handle common aliases for non-tested reagents.
    Fail closed when unknown, instead of guessing from textual similarity.
    """
    common = {
        "DsbC (E. coli, 25.5 kDa)": "dsbc_ecoli",
        "PEG-8000": "peg_8000",
        "Magnesium Glutamate": "magnesium_glutamate",
        "HEPES-KOH pH 7.5": "hepes_koh",
        "Coenzyme A": "coa",
        "PEP mono (0.1M stock)": "pep_mono",
        "17 Amino Acid Mix": "amino_acid_mix_17",
        "Nicotinamide": "nicotinamide",
        "Glucose": "glucose",
        "Folinic Acid": "folinic_acid",
        "Potassium Glutamate": "potassium_glutamate",
        "Potassium Oxalate": "potassium_oxalate",
        "Ammonium Glutamate": "ammonium_glutamate",
        "Reduced Glutathione (GSH)": "reduced_glutathione",
        "Oxidized Glutathione (GSSG)": "oxidized_glutathione",
        "Maltodextrin 17": "maltodextrin_17",
        "Spermidine": "spermidine",
        "Tyrosine pH 12": "tyrosine",
        "Potassium phosphate 1.6:1 dibasic:monobasic (molar ratio)": "kpo_dibasic_mix",
        "Potassium phosphate 1.6:1 monobasic:dibasic (molar ratio)": "kpo_monobasic_mix",
        "Putrescine Dihydrochloride": "putrescine",
    }
    for category in ("neverOnAPlate", "pastThePlatesRange"):
        for d in record.get("vsGinkgoPlates", {}).get(category, []):
            if d.get("name") == ingredient:
                return str(d["reagent"])
    if ingredient in common:
        return common[ingredient]
    slug = ingredient.lower().replace("-", "_").replace(" ", "_")
    if slug in record.get("recipeNl", {}):
        return slug
    for reagent in record.get("recipeNl", {}):
        if str(reagent).casefold() == ingredient.casefold():
            return reagent
    return ""  # Caller must flag unresolved; no inferential fallback.


def _validated_records(data: Mapping[str, Any]) -> tuple[dict[str, dict[str, dict]], list[dict]]:
    if not isinstance(data, Mapping) or not isinstance(data.get("proteins"), Mapping):
        raise ValueError("Expected published clusters_designs.json with proteins mapping")
    by_target = {}
    rows = []
    global_seen = set()
    for target in sorted(data["proteins"]):
        block = data["proteins"][target]
        designs = block.get("designs")
        if not isinstance(designs, list):
            raise ValueError(f"Missing designs list for {target}")
        lookup = {}
        for design in designs:
            rid = str(design["id"])
            if rid in lookup or rid in global_seen:
                raise ValueError(f"Nonunique design ID: {rid}")
            global_seen.add(rid)
            if design.get("protein") != target:
                raise ValueError(f"Design {rid} target mismatch")
            recipe = design.get("recipeNl")
            key = canonical_recipe(recipe)
            total = sum(float(v) for v in recipe.values())
            lookup[rid] = design
            rows.append({
                "protein": target, "id": rid,
                "sampleId": design.get("sampleId"), "author": design.get("author"),
                "round1Role": design.get("round1Role"),
                "selectedProposed": bool(design.get("round1Role") in BASELINE_ROLES),
                "sameRecipeAs": design.get("sameRecipeAs"),
                "group": (design.get("group") or {}).get("letter"),
                "recipe_key": key,
                "recipe_hash": hashlib.sha256(json.dumps(key).encode()).hexdigest(),
                "total_nL": total,
                "total_valid": abs(total - REACTION_NL) < 1e-5,
                "n_active_reagents": len(key),
                "pBeat_model": (design.get("forecast") or {}).get("pBeat"),
                "lnCostPerGram_model": (design.get("forecast") or {}).get("lnSpecCost"),
            })
        by_target[target] = lookup
    return by_target, rows


def _sweep_rows(data: Mapping[str, Any], by_target: dict, selected_ids: dict) -> list[dict]:
    result = []
    for target in sorted(by_target):
        lookup = by_target[target]
        for ingredient_block in data["proteins"][target].get("sweeps", []):
            factor = ingredient_block["reagent"]
            for family_index, sweep in enumerate(ingredient_block.get("sweeps", [])):
                to_run = sweep.get("toRun") or []
                if not to_run:
                    continue
                ids = [x.get("id") if isinstance(x, Mapping) else x for x in to_run]
                ids = [str(x) for x in ids]
                missing = [rid for rid in ids if rid not in lookup]
                if missing:
                    raise ValueError(f"Unknown IDs in sweep {target}/{factor}: {missing}")
                unique_selected = {}
                background = set()
                for rid in ids:
                    if rid not in selected_ids[target]:
                        continue
                    design = lookup[rid]
                    unique_selected[canonical_recipe(design["recipeNl"])] = design
                    background.add(signature_except(design["recipeNl"], factor))
                levels = sorted({float(r["recipeNl"].get(factor, 0)) for r in unique_selected.values()})
                result.append({
                    "protein": target, "factor": factor, "family_index": family_index,
                    "recommendedByModel": bool(ingredient_block.get("recommended")),
                    "n_toRun_ids": len(ids), "n_toRun_unique": len(unique_selected),
                    "n_selected_levels": len(levels), "selected_levels_nL": ";".join(map(lambda x: str(int(x)) if x.is_integer() else str(x), levels)),
                    "background_signature_count": len(background),
                    "clean_single_factor_sweep": len(background) == 1,
                    "at_least_three_levels": len(levels) >= 3,
                    "run_ids": ";".join(ids),
                })
    return result


def _cluster_checks(data: Mapping[str, Any], by_target: dict, selected_ids: dict) -> tuple[list[dict], list[dict]]:
    checks, additions = [], []
    for target in sorted(by_target):
        lookup = by_target[target]
        for group in data["proteins"][target].get("groups", []):
            base_id = str(group["firstRun"])
            if base_id not in lookup:
                raise ValueError(f"Unknown first run: {base_id}")
            base = lookup[base_id]
            for check in group.get("checkRuns", []):
                check_id = str(check["id"])
                if check_id not in lookup:
                    raise ValueError(f"Unknown check run: {check_id}")
                design = lookup[check_id]
                factor = _factor_key_from_display(check["ingredient"], design)
                if not factor:
                    # Try first-run fields, as named ingredient might be absent in check
                    factor = _factor_key_from_display(check["ingredient"], base)
                changed = changed_reagents(base["recipeNl"], design["recipeNl"])
                named_changed = factor in changed if factor else False
                exact_selected = []
                exact_unselected = []
                if factor:
                    sig = signature_except(design["recipeNl"], factor)
                    for other_id, other in lookup.items():
                        if other_id == check_id:
                            continue
                        if signature_except(other["recipeNl"], factor) != sig:
                            continue
                        if np.isclose(recipe_factor_value(other["recipeNl"], factor), recipe_factor_value(design["recipeNl"], factor)):
                            continue
                        (exact_selected if other_id in selected_ids[target] else exact_unselected).append(other_id)
                exact_selected = sorted(exact_selected)
                exact_unselected = sorted(exact_unselected)
                checks.append({
                    "protein": target, "group": group["letter"],
                    "firstRunId": base_id, "checkRunId": check_id,
                    "namedFactorLabel": check["ingredient"], "namedFactor": factor,
                    "named_factor_changed": named_changed,
                    "changed_factor_count": len(changed),
                    "changed_factors": ";".join(changed),
                    "exact_one_factor_with_first_run": len(changed) == 1 and named_changed,
                    "any_selected_exact_counterpart": bool(exact_selected),
                    "any_unselected_exact_counterpart": bool(exact_unselected),
                    "model_link_with_first": (design.get("group") or {}).get("linkWithFirstRun"),
                    "model_new_information_pct": check.get("newInformation"),
                    "model_price_only": check.get("priceOnly"),
                })
                if not exact_selected and exact_unselected and factor:
                    # Stable deterministic candidate: prefer the largest dose contrast, then lowest ID.
                    own_level = recipe_factor_value(design["recipeNl"], factor)
                    chosen = sorted(exact_unselected, key=lambda rid: (-abs(recipe_factor_value(lookup[rid]["recipeNl"], factor) - own_level), rid))[0]
                    additions.append({
                        "protein": target, "group": group["letter"],
                        "checkRunId": check_id, "namedFactor": factor,
                        "checkFactorNl": own_level, "candidateId": chosen,
                        "candidateFactorNl": recipe_factor_value(lookup[chosen]["recipeNl"], factor),
                        "is_exact_one_factor_against_check": True,
                    })
    return checks, additions


@dataclass
class AuditResult:
    summary: dict
    candidates: pd.DataFrame
    duplicates: pd.DataFrame
    check_runs: pd.DataFrame
    sweep_families: pd.DataFrame
    additional_contrasts: pd.DataFrame
    records: dict[str, dict[str, dict]]

    def save(self, out: str | Path, *, input_paths: Sequence[str | Path] = ()) -> dict:
        """Write private row-level CSVs and provenance manifest.

        The `publish_row_level` flag requires explicit opt-in; caller controls actual
        directory permissions. No code uploads or pushes files to Git automatically.
        """
        out = Path(out)
        out.mkdir(parents=True, exist_ok=True)
        exports = {
            "candidate_audit.csv": self.candidates.drop(columns=["recipe_key"], errors="ignore"),
            "duplicate_formulations.csv": self.duplicates,
            "cluster_check_audit.csv": self.check_runs,
            "sweep_family_audit.csv": self.sweep_families,
            "additional_exact_contrasts.csv": self.additional_contrasts,
        }
        for filename, frame in exports.items():
            frame.to_csv(out / filename, index=False)
        with open(out / "audit_summary.json", "w") as fh:
            json.dump(self.summary, fh, indent=2, sort_keys=True, allow_nan=False)
        manifest = {
            "schema": "cfps-selection-audit/v1", "analysis_version": VERSION,
            "input_sha256": {Path(p).name: input_sha256(p) for p in input_paths},
            "working_quota": self.summary["working_quota_per_protein"],
            "python": __import__("sys").version.split()[0],
            "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": __import__("scipy").__version__,
            "parameters": {
                "reaction_nL": REACTION_NL,
                "baseline_roles": sorted(BASELINE_ROLES),
                "duplicate_recipe_excludes": sorted(EXCLUDED_BALANCERS),
                "recipe_volume_unit": "nL (stock dispensing volumes, not molarity)",
                "selected_unique_rule": "one exact dispensing formulation per target",
            },
            "files": sorted(exports) + ["audit_summary.json"],
            "classification": "row-level competition data; private until approved",
            "publication_review_required": True,
        }
        (out / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        return manifest


def audit_gp_model_snapshot(data: Mapping[str, Any], *, working_quota: int = WORKING_QUOTA) -> AuditResult:
    """Deterministic end-to-end audit; never modifies raw data or model forecasts."""
    if int(working_quota) < 1:
        raise ValueError("working_quota must be positive")
    by_target, rows = _validated_records(data)
    candidates = pd.DataFrame(rows).sort_values(["protein", "id"], kind="stable").reset_index(drop=True)
    # Internal exact-key maps; no cross-target merging.
    selected_ids = {}
    duplicates = []
    summary_proteins = {}
    for target in sorted(by_target):
        part = candidates.loc[candidates["protein"] == target].copy()
        selected = part.loc[part["selectedProposed"]]
        all_groups = defaultdict(list)
        for r in part.itertuples(index=False):
            all_groups[r.recipe_key].append(r.id)
        selected_groups = defaultdict(list)
        for r in selected.itertuples(index=False):
            selected_groups[r.recipe_key].append(r.id)
        selected_ids[target] = set(selected["id"])
        for signature, ids in all_groups.items():
            if len(ids) > 1:
                duplicates.append({
                    "protein": target, "count": len(ids), "ids": ";".join(sorted(ids)),
                    "canonical_signature_sha256": hashlib.sha256(repr(signature).encode()).hexdigest(),
                    "selected_proposed_count": sum(rid in selected_ids[target] for rid in ids),
                })
        block = data["proteins"][target]
        untested_groups = sum(bool(g.get("linkFromSharedUntested", 0) > 0) for g in block.get("groups", []))
        summary_proteins[target] = {
            "candidateDesigns": int(len(part)),
            "distinctFormulations": int(len(all_groups)),
            "duplicateExcess": int(len(part) - len(all_groups)),
            "selectedRoleRecords": int(len(selected)),
            "selectedUniqueFormulations": int(len(selected_groups)),
            "selectedDuplicateExcess": int(len(selected) - len(selected_groups)),
            "gapToWorkingQuota": int(working_quota - len(selected_groups)),
            "roleBreakdownUniqueDesignIDs": {str(k): int(v) for k,v in selected["round1Role"].value_counts().sort_index().items()},
            "groups": len(block.get("groups", [])),
            "groupsWithSharedUntested": int(untested_groups),
            "authorsAll": int(part["author"].nunique(dropna=True)),
            "authorsSelected": int(selected["author"].nunique(dropna=True)),
            "volumeValidRecords": int(part["total_valid"].sum()),
            "sourceTotals": block.get("totals", {}),
        }
    checks, add = _cluster_checks(data, by_target, selected_ids)
    sweeps = _sweep_rows(data, by_target, selected_ids)
    check_cols = ["protein", "group", "firstRunId", "checkRunId", "namedFactorLabel", "namedFactor", "named_factor_changed", "changed_factor_count", "changed_factors", "exact_one_factor_with_first_run", "any_selected_exact_counterpart", "any_unselected_exact_counterpart", "model_link_with_first", "model_new_information_pct", "model_price_only"]
    sweep_cols = ["protein", "factor", "family_index", "recommendedByModel", "n_toRun_ids", "n_toRun_unique", "n_selected_levels", "selected_levels_nL", "background_signature_count", "clean_single_factor_sweep", "at_least_three_levels", "run_ids"]
    add_cols = ["protein", "group", "checkRunId", "namedFactor", "checkFactorNl", "candidateId", "candidateFactorNl", "is_exact_one_factor_against_check"]
    checks_df = pd.DataFrame(checks,columns=check_cols).sort_values(["protein", "group", "checkRunId"]).reset_index(drop=True)
    sweeps_df = pd.DataFrame(sweeps,columns=sweep_cols).sort_values(["protein", "factor", "family_index"]).reset_index(drop=True)
    add_df = pd.DataFrame(add,columns=add_cols).sort_values(["protein", "group", "checkRunId"]).reset_index(drop=True)
    for target in summary_proteins:
        c = checks_df[checks_df["protein"] == target]
        s = sweeps_df[sweeps_df["protein"] == target]
        a = add_df[add_df["protein"] == target]
        summary_proteins[target].update({
            "clusterChecks": int(len(c)),
            "exactOneFactorChecksVsFirst": int(c["exact_one_factor_with_first_run"].sum()),
            "multifactorChecksVsFirst": int((c["changed_factor_count"] > 1).sum()),
            "unresolvedNamedFactorChecks": int((c["namedFactor"] == "").sum()),
            "newExactCounterpartsForChecks": int(len(a)),
            "listedSweepFamiliesWithRuns": int(len(s)),
            "cleanSweepFamilies": int(s["clean_single_factor_sweep"].sum()),
            "sweepFamiliesWithThreeOrMoreLevels": int(s["at_least_three_levels"].sum()),
        })
    duplicate_df = pd.DataFrame(duplicates).sort_values(["protein", "canonical_signature_sha256"]).reset_index(drop=True)
    summary = {
        "analysis_version": VERSION,
        "source_built_utc": data.get("built"),
        "scope": "audit of published GP-informed proposed runs, not measured results or final selection",
        "working_quota_per_protein": int(working_quota),
        "proteins": summary_proteins,
        "interpretation": {
            "model_based": "forecast, expected links and LHS sensitivities are pre-run model products",
            "model_free": "recipe identities, exact contrasts, sweep completeness, distance/coverage",
            "price_per_gram": "model forecast; not validated observed cost per gram",
            "experimental_info_gain": "NOT calculated: model newInformation is a heuristic",
        },
    }
    return AuditResult(summary, candidates, duplicate_df, checks_df, sweeps_df, add_df, by_target)


def read_gp_model_file(path: str | Path) -> dict:
    with open(path, "rb") as fh:
        return json.load(fh)


def _variable_recipe_matrix(rows: list[dict], *, feature_mode: str = "scaled_volume", presence_weight: float = 0.35) -> tuple[np.ndarray, list[str]]:
    """Candidate rows -> scaled recipe matrix; no model predictions involved.

    Excludes water and columns invariant in this protein. For concentrations,
    use stock nL only as a per-reagent within-stock proxy, NOT chemical molarity.
    Normalize by max observed nL (>=1) to make each ingredient dimension comparable.
    """
    if feature_mode not in {"scaled_volume", "presence", "scaled_plus_presence"}:
        raise ValueError("Invalid feature_mode")
    all_recipes = [r["recipeNl"] for r in rows]
    keys = sorted({k for r in all_recipes for k in r if k not in EXCLUDED_BALANCERS})
    A = np.asarray([[float(rec.get(k, 0)) for k in keys] for rec in all_recipes], dtype=float)
    var = np.ptp(A, axis=0) > 1e-9
    A = A[:, var]
    keys = [k for k, keep in zip(keys, var) if keep]
    if not keys:
        raise ValueError("No variable reagent dimensions")
    scales = np.maximum(np.max(A, axis=0), 1)
    Z = A / scales
    if feature_mode == "presence":
        return (A > 0).astype(float), keys
    if feature_mode == "scaled_plus_presence":
        Z = np.column_stack([Z, (A > 0).astype(float)*math.sqrt(presence_weight)])
        return Z, keys + ["presence:"+k for k in keys]
    return Z, keys


def coverage_metrics(candidate_rows: list[dict], selected_ids: set[str], *, metric: str = "euclidean") -> dict:
    """Model-free dispersion/coverage diagnostics of an existing-recipe portfolio.

    A candidate-population metric: no claim of uniformity in unobserved chemical space.
    """
    selected_indices = [i for i,r in enumerate(candidate_rows) if r["id"] in selected_ids]
    if not selected_indices:
        raise ValueError("No selected IDs in candidate population")
    Z, keys = _variable_recipe_matrix(candidate_rows, feature_mode="scaled_plus_presence")
    ds = cdist(Z, Z[selected_indices], metric=metric)
    nearest = ds.min(axis=1)
    # Unique physical formulations, not student submission records, are the
    # scientific sampling units. Equal-weight each distinct recipe.
    signatures = [canonical_recipe(r["recipeNl"]) for r in candidate_rows]
    representative_indices = list({sig:i for i,sig in reversed(list(enumerate(signatures)))}.values())
    unique_nearest = nearest[representative_indices]
    factors = {}
    for k in sorted({k for r in candidate_rows for k in r["recipeNl"] if k not in EXCLUDED_BALANCERS}):
        all_values = np.array([float(r["recipeNl"].get(k,0)) for r in candidate_rows])
        sel_values = all_values[selected_indices]
        unique_positive = sorted(set(all_values[all_values>0]))
        # Bins drawn from candidate quantiles: a coverage *diagnostic*, not a perfect LHS.
        if len(unique_positive) >= 2:
            bounds = np.unique(np.quantile(unique_positive, np.linspace(0,1,6)))
            pos_all = np.digitize(all_values[all_values>0], bounds[1:-1])
            pos_sel = np.digitize(sel_values[sel_values>0], bounds[1:-1])
            all_bins = set(pos_all)
            sel_bins = set(pos_sel)
            factor_coverage = len(sel_bins & all_bins)/len(all_bins) if all_bins else 1.0
        else:
            factor_coverage = 1.0 if (len(unique_positive)==0 or (sel_values>0).any()) else 0.0
        factors[k] = {"candidate_bin_coverage": round(float(factor_coverage),6),
                      "has_absent": bool((sel_values==0).any()),
                      "has_present": bool((sel_values>0).any())}
    return {
        "n_selected_ids": len(selected_indices),
        "n_distinct_candidate_formulations": len(representative_indices),
        "feature_mode": "scaled_plus_presence", "n_features": Z.shape[1],
        "mean_candidate_to_selected_distance": float(unique_nearest.mean()),
        "p90_candidate_to_selected_distance": float(np.quantile(unique_nearest, 0.90)),
        "max_candidate_to_selected_distance": float(unique_nearest.max()),
        "factor_coverage": factors,
    }


def suggest_portfolio(
    candidate_rows: list[dict], *, quota: int = WORKING_QUOTA,
    mode: str = "hybrid", objective: str = "facility",
    seed_ids: Sequence[str] = (),
    exact_unique: bool = True,
) -> dict:
    """Deterministic proposed IDs. No new recipes; no GP calls.

    - 'gp_model_baseline': return the original GP-informed proposed selection *without padding*.
    - 'hybrid': lock the published GP-informed baseline plus explicitly given seeds, fill to quota.
    - 'model_free': ignore model-based roles/predictions, use only seeds and geometry.

    A *diagnostic selection* only. Does not guarantee experimental contrast quality,
    student fairness, batch feasibility, or a scientifically optimal design.
    """
    if mode not in {"gp_model_baseline", "hybrid", "model_free"}:
        raise ValueError("mode must be gp_model_baseline/hybrid/model_free")
    if objective not in {"facility", "maximin"}:
        raise ValueError("objective must be facility/maximin")
    if quota < 1:
        raise ValueError("quota must be positive")
    ordered = sorted(candidate_rows, key=lambda r:r["id"])
    idmap = {r["id"]:i for i,r in enumerate(ordered)}
    if len(idmap) != len(ordered):
        raise ValueError("Non-unique IDs")
    # Published export labels include multiple IDs for the *same* dispensing formulation.
    # Physical baseline counts are unique chemical recipes; preserve provenance via
    # duplicate-group output, and prefer a first run, check, sweep, independent.
    priority = {"first run": 0, "check run": 1, "sweep run": 2, "independent run": 3}
    reps = {}
    for r in sorted(ordered, key=lambda x:(priority.get(x["round1Role"],99),x["id"])):
        if r["round1Role"] in BASELINE_ROLES:
            reps.setdefault(canonical_recipe(r["recipeNl"]), r["id"])
    baseline = set(reps.values())
    if mode == "gp_model_baseline":
        return {"mode":mode,"objective": "baseline", "ids": sorted(baseline),
                "count":len(baseline),"quota_gap":quota-len(baseline)}
    selected = set(seed_ids)
    if unknown := selected-set(idmap):
        raise ValueError(f"Unknown seeds: {sorted(unknown)}")
    if mode == "hybrid":
        already_represented = {canonical_recipe(ordered[idmap[rid]]["recipeNl"]) for rid in selected}
        selected.update(rid for signature,rid in reps.items() if signature not in already_represented)
    if len(selected) > quota:
        raise ValueError(f"Selected seeds/baseline exceed quota {quota}")
    # Identity buckets: even when records have separate IDs, avoid paying twice for same recipe.
    keys = [canonical_recipe(r["recipeNl"]) for r in ordered]
    represented = {keys[idmap[rid]] for rid in selected}
    if exact_unique and len(represented) < len(selected):
        raise ValueError("Seed/baseline contains chemically identical formulations")
    allowed = [i for i, r in enumerate(ordered) if (not exact_unique or keys[i] not in represented)]
    Z, feature_names = _variable_recipe_matrix(ordered, feature_mode="scaled_plus_presence")
    dist = cdist(Z, Z, metric="euclidean")
    # Equal weight to distinct formulations; otherwise students submitting
    # identical recipes many times would bias coverage optimization.
    signature_count = Counter(keys)
    weights = np.asarray([1.0 / signature_count[k] for k in keys])
    weights = weights / weights.sum()
    # Initial seed-free selection: medoid minimizes total candidate distance for facility,
    # maximum total distance for maximin (farthest extremum; documented heuristic).
    best_distance = np.min(dist[:,[idmap[i] for i in selected]], axis=1) if selected else None
    while len(selected)<quota and allowed:
        if best_distance is None:
            scores = weights @ dist
            pick = min(allowed, key=lambda i:(scores[i],ordered[i]["id"])) if objective=="facility" else max(allowed,key=lambda i:(scores[i],ordered[i]["id"]))
        elif objective=="maximin":
            pick = max(allowed, key=lambda i:(float(best_distance[i]),ordered[i]["id"]))
        else:
            # Weighted candidate coverage: minimize sum of nearest distances.
            pick = min(allowed,key=lambda i:(float(weights @ np.minimum(best_distance,dist[:,i])),ordered[i]["id"]))
        chosen_id = ordered[pick]["id"]
        selected.add(chosen_id)
        represented.add(keys[pick])
        best_distance = dist[:,pick].copy() if best_distance is None else np.minimum(best_distance,dist[:,pick])
        allowed = [i for i in allowed if (i != pick and (not exact_unique or keys[i] not in represented))]
    return {"mode":mode,"objective":objective,"ids":sorted(selected),
            "count":len(selected), "quota_gap":quota-len(selected),
            "n_features":len(feature_names),
            "coverage_mean":float(weights @ best_distance) if best_distance is not None else None,
            "selection_status":"PROVISIONAL: geometry-only fill; controls/contrasts/fairness require separate review"}


def write_selection_overlay(
    audit: AuditResult, portfolios: Mapping[str, Mapping[str, dict]], path: str | Path,
    *, include_private_ids: bool = False,
) -> None:
    """Precomputed JSON for an opt-in browser visualization; author names omitted.

    ID-level overlay remains private by default; explicit approval before publishing.
    """
    result = {"schema":"cfps-selection-overlay/v1", "audit_version":VERSION,
              "snapshot_built":audit.summary["source_built_utc"], "targets":{}}
    for target in sorted(audit.records):
        port = portfolios.get(target, {})
        pitems = {}
        for label, d in port.items():
            pitems[label] = {k:v for k,v in d.items() if k not in {"ids"}}
            if include_private_ids:
                pitems[label]["ids"] = d.get("ids",[])
        result["targets"][target] = {
            "audit": {k:v for k,v in audit.summary["proteins"][target].items() if k!="sourceTotals"},
            "portfolios":pitems,
        }
    Path(path).write_text(json.dumps(result,indent=2,sort_keys=True,allow_nan=False)+"\n",encoding="utf-8")


def audit_gp_model_link_matrices(
    links: Mapping[str, Any] | str | Path, *, design_audit: AuditResult | None = None
) -> dict:
    """Audit *published/displayed* link matrices (not estimated GP covariance).

    For cross-snapshot validity, check matching row counts separately; no IDs are
    inferred from the rendered `names` strings. A rounded correlation-like matrix
    with negative eigenvalues should NOT be used as a GP covariance without an
    explicitly justified PSD repair and model verification.
    """
    if isinstance(links, (str, Path)):
        p = Path(links)
        with (gzip.open(p, "rt", encoding="utf-8") if p.suffix == ".gz" else p.open("r", encoding="utf-8")) as f:
            data = json.load(f)
    else:
        data = links
    if not isinstance(data, Mapping) or "tips" not in data:
        raise ValueError("Expected published clusters.json.gz with 'tips'")
    result = {"kind":"DISPLAY link-matrix audit, not biological outcome correlation",
              "built":data.get("built"), "targets":{}}
    names = {"sfgfp":"sfGFP", "petase":"PETase", "reteplase":"Reteplase"}
    for key, tip in sorted(data["tips"].items()):
        A = np.asarray(tip["C"],dtype=float)
        if A.ndim != 2 or A.shape[0] != A.shape[1] or not np.isfinite(A).all():
            raise ValueError(f"Invalid link matrix for {key}")
        n=A.shape[0]
        symmetric_max=float(np.max(np.abs(A-A.T)))
        B=(A+A.T)/2.0
        eig=np.linalg.eigvalsh(B)
        target=names.get(key,key)
        expected = (design_audit.summary["proteins"].get(target) or {}).get("candidateDesigns") if design_audit else None
        result["targets"][target] = {
            "n_rows":int(n),"n_names":len(tip.get("names",[])),
            "candidate_count_matches":None if expected is None else bool(n==expected),
            "entry_min":float(A.min()),"entry_max":float(A.max()),
            "diagonal_max_abs_error_from_one":float(np.max(np.abs(np.diag(A)-1))),
            "max_absolute_asymmetry":symmetric_max,
            "min_eigenvalue_of_symmetrized_DISPLAY_matrix":float(eig[0]),
            "negative_eigenvalue_count_below_minus_1e_6":int((eig < -1e-6).sum()),
            "PSD_as_stored_tolerance_1e_6": bool(eig[0]>=-1e-6),
            "interpretation":"Display values rounded; not independently verified GP covariance",
        }
    if design_audit is not None:
        check=design_audit.check_runs.dropna(subset=["model_link_with_first","model_new_information_pct"])
        if len(check):
            v=np.asarray(check["model_link_with_first"],dtype=float)
            scores=np.asarray(check["model_new_information_pct"],dtype=float)
            deviations=np.abs(scores-100*(1-v**2))
            result["newInformation_heuristic_check"] = {
                "n_checked":int(len(check)),
                "max_absolute_pct_point_difference":float(np.max(deviations)),
                "median_absolute_pct_point_difference":float(np.median(deviations)),
                "tested_formula":"100 * (1 - displayed_link_to_first**2)",
                "not_expected_information_gain":True,
            }
    return result


In [ ]:
# 14. Run pre-experimental audit from a local export without requiring cloned modules.
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
if RUN_GP_BASELINE_AUDIT:
    gp_snapshot_path = Path(GP_DESIGNS_PATH).expanduser()
    if not gp_snapshot_path.exists():
        raise FileNotFoundError(
            f"Upload clusters_designs.json into Colab's Files panel and set GP_DESIGNS_PATH. "
            f"Current path: {gp_snapshot_path}"
        )
    gp_data = read_gp_model_file(gp_snapshot_path)
    audit = audit_gp_model_snapshot(gp_data, working_quota=WORKING_QUOTA_NOTEBOOK)
    audit_out = PRIVATE_ROOT / ("gp_model_audit_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    gp_links_path = Path(GP_LINKS_PATH).expanduser()
    audit_manifest = audit.save(audit_out, input_paths=[gp_snapshot_path]+([gp_links_path] if gp_links_path.exists() else []))
    if gp_links_path.exists():
        link_summary = audit_gp_model_link_matrices(gp_links_path, design_audit=audit)
        (audit_out/"model_link_matrix_audit.json").write_text(json.dumps(link_summary,indent=2),encoding="utf-8")
        print("Published link-matrix audit:", {k:v["min_eigenvalue_of_symmetrized_DISPLAY_matrix"] for k,v in link_summary["targets"].items()})
    audit_summary_df = pd.DataFrame([
        {"protein":name, "candidate_count":x["candidateDesigns"],
         "distinct_formulations":x["distinctFormulations"],
         "gp_baseline_unique_runs":x["selectedUniqueFormulations"],
         "quota_gap":x["gapToWorkingQuota"],
         "exact_one_factor_checks":x["exactOneFactorChecksVsFirst"],
         "additional_exact_matches":x["newExactCounterpartsForChecks"]}
        for name,x in sorted(audit.summary["proteins"].items())
    ])
    display(audit_summary_df)
    print("SHA256", audit_manifest["input_sha256"])
    print("PRIVATE output:", audit_out)
    if "snapshot" in globals() and isinstance(snapshot,dict):
        live_ids = {str(r.get("id")) for r in snapshot["collections"].get("cfps_designs",[])}
        gp_ids = {rid for block in audit.records.values() for rid in block}
        crosswalk = {"shared_design_ids":len(live_ids & gp_ids),
                     "live_only_ids":len(live_ids-gp_ids),
                     "gp_export_only_ids":len(gp_ids-live_ids)}
        print("Snapshot crosswalk (potentially different snapshot times):",crosswalk)

In [ ]:
# 15. Diagnostic selection modes on ACTUAL recipe IDs (opt-in; not final selection).
if RUN_GP_BASELINE_AUDIT and RUN_PORTFOLIO_PREVIEWS:
    from copy import deepcopy
    preview = {}
    for target in sorted(audit.records):
        rows = [audit.records[target][rid] for rid in sorted(audit.records[target])]
        preview[target] = {}
        for mode in ("gp_model_baseline", "hybrid", "model_free"):
            alternative = suggest_portfolio(
                rows, quota=WORKING_QUOTA_NOTEBOOK, mode=mode, objective=GEOMETRY_OBJECTIVE,
            )
            diagnostics = coverage_metrics(rows, set(alternative["ids"]))
            preview[target][mode] = alternative
            print(target,mode,"n:", alternative["count"],
                  "mean coverage distance:", round(diagnostics["mean_candidate_to_selected_distance"],5))
    write_selection_overlay(audit, preview, audit_out/"selection_overlay_private_summary.json")
    (audit_out/"selection_ids_private.json").write_text(json.dumps(
        {target:{mode:item["ids"] for mode,item in modes.items()}
         for target,modes in preview.items()}, indent=2), encoding="utf-8")
    print("Provisional candidate IDs saved privately; do not submit without controls/contrast review.")

## 16. Exact mathematical interpretation and decision framework

**GP-informed model-based approach.** A protein-specific Gaussian-process (GP)
regression is fitted on Ginkgo's measured control wells, schematically
\(f_p(x)\sim\mathcal{GP}(m_p(x),k_p(x,x'))\). For each submitted recipe \(x_i\),
the GP yields predicted output and uncertainty for the *plate-tested* features.
The price \(C(x_i)\) comes from the reagent price schedule. His model projects
cost per gram \(C(x_i)/\hat Y_p(x_i)\), with additional **assumed**, not
measured, risks for untested ingredients. His expected co-movement links combine
recipe similarity and shared untested risks; those links are not empirical
correlations or guaranteed covariance matrices.

**What LHS actually does.** Within a cluster, ≥160 **virtual** formulations
sample factor ranges with Latin hypercube stratification; each is passed to the
*existing model*, and a second regression approximates the model's predictions
over that region, identifying influential factors. GP-informed then picks **actual
submitted recipes** as checks that differ on those factors, considering predicted
odds. This is *not* established to be a nearest-neighbor projection of virtual
recipes to submitted ones, nor a claim that LHS generates lab-ready candidates.
It tests the model's sensitivities rather than establishing causal effects.

**Your model-free complement.** For existing recipes \(x_i\), omit water and
constant reagent dimensions, scale each variable stock-volume dimension by its
maximum observed value (to be replaced with chemically meaningful molar limits
when verified), and optionally append an ingredient-presence indicator. Then:

- Greedy **maximin** fill: \(x^*=\arg\max_{x\in X\\S}\min_{s\in S} d(x,s)\).
- Greedy **facility-location** fill: choose \(x^*\) minimizing
  \(\sum_{u\in X}\min(d(u,x),\min_{s\in S}d(u,s))\).
- *Exact matched contrast* exists when two recipes vary in exactly one
  non-water ingredient; a clean sweep has a common non-swept background
  and at least three distinct levels.

Three configurable strategies are included: `gp_model_baseline` (model-driven baseline),
`hybrid` (baseline locked, geometry fills quota), `model_free` (no GP roles or
forecasts used). **These strategies are provisional computational baselines.**
The final portfolio requires independent review of controls, design validity,
sweep coverage, author representation, quota rules, and biochemical interactions.

Critically, facility location covers the **observed candidate distribution**, not
all feasible CFPS chemistry; maximin can emphasize outliers. A distance-based
output is not a formal expected information-gain calculation.

### Reproducibility requirements

Keep frozen `clusters_designs.json`, input hash, code snapshot, raw Ginkgo model
dependencies if attempting to reproduce the published GP-informed GP, deterministic tie-breaking,
audit version, and dependency versions. Raw ID-level outputs remain outside Git.
The current notebook does **not** reproduce the published GP-informed fitted GP, its error
covariance, virtual LHS seed, or model-risk hyperparameters from first principles;
those would require his original training data and full source.

In [ ]:
# 16.1 Snapshot-specific regression checks. Adapt explicitly if the source data change.
if RUN_GP_BASELINE_AUDIT:
    EXPECTED_FILE_SHA256 = "0c312d2a4c2112e5319d988017d69c10a19bf8c5688b01ef190707ffb6ca48f6"
    if input_sha256(gp_snapshot_path) == EXPECTED_FILE_SHA256:
        expected = {
            "sfGFP": (549,484,318,18,5),
            "PETase": (500,459,284,25,10),
            "Reteplase": (551,488,179,18,3),
        }
        for protein,(n,distinct,chosen,one_factor,new_pairs) in expected.items():
            p = audit.summary["proteins"][protein]
            observed=(p["candidateDesigns"],p["distinctFormulations"],p["selectedUniqueFormulations"],
                      p["exactOneFactorChecksVsFirst"],p["newExactCounterpartsForChecks"])
            assert observed==(n,distinct,chosen,one_factor,new_pairs), (protein,observed)
        print("PASS: known-input regression checks")
    else:
        print("Source differs from the pinned data SHA256: review revised dataset and version outputs.")

## 17. Deliverables, permissions and sources

This notebook is **standalone**: the correlation and selection functions are
*defined in executable cells*, so the notebook never imports mirrored project `.py`
modules. The repository distribution includes importable modules and synthetic unit
tests for headless analyses. The reproducibility manifest records the data hash,
runtime dependencies, version and normalization choices.

- Notebook source: `notebooks/cfps_database_snapshot_and_exploration.ipynb`
- Modules: `analysis/cfps_correlation/pairwise_correlation_clustering.py`,
  `analysis/cfps_selection/selection_framework.py`
- CLI: `tools/audit_gp_model_baseline.py`
- Tests: `python -m unittest discover -s tests -v`

No Git operations, private data publishing, or competition submissions occur automatically.
The `selection_overlay_private_summary.json` is a preliminary scaffold for later
integration into the published GP-informed website; his `index.md`, loader, and stylesheet should
only be edited after reviewing the actual visualization source and permissions.


## 18. Source normalization and provenance

Choose `INPUT_SOURCE` in the first configuration cell. `published_snapshot` uses the
GP-informed `clusters_designs.json` provided with the analysis; `pocketbase_snapshot`
uses a previously authorized `pocketbase_snapshot.json`; `pocketbase_live` first fetches
a read-only PocketBase snapshot using Colab Secrets. Both PocketBase modes normalize
`cfps_designs.design_json` into an analysis-compatible *experimental-recipe pool*.

**No forecasts or selection annotations are fabricated.** PocketBase-only runs enable
`model_free` selection; the historical GP-informed baseline and hybrid are unavailable
unless a validated GP-annotated export is analyzed separately. Non-experimental records
are excluded and counted, while malformed experimental records cause an error.
Inputs and normalized copies are private; nominally 20 µL volumes are checked by the
audit, not chemically inferred from stock names.


In [ ]:
"""Validated participant-pool input adapters for CFPS design selection.

Accepted inputs:
- ``published_snapshot``: the dated GP-informed ``clusters_designs.json`` export;
- ``pocketbase_snapshot``: an authorized, read-only PocketBase snapshot containing
  ``collections.cfps_designs`` or the corresponding paginated records payload.

The PocketBase adapter deliberately does not infer GP forecasts, cluster membership,
run roles, experimental output, or chemical stock concentrations. Consequently,
GP-informed baseline and hybrid modes must not run on PocketBase-only input.

This module is embedded as an independent code cell in the Colab notebook.
"""
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any, Mapping

SOURCE_ADAPTER_VERSION = "1.1.0"
SOURCE_TYPES = ("published_snapshot", "pocketbase_snapshot")
CANONICAL_PROTEINS = ("sfGFP", "PETase", "Reteplase")


def _json_object(value: Any, label: str) -> dict:
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (ValueError, TypeError) as exc:
            raise ValueError(f"{label} is not valid JSON") from exc
    if not isinstance(value, Mapping):
        raise ValueError(f"{label} must be a JSON object")
    return dict(value)


def _canonical_target(value: Any) -> str:
    text = str(value or "").strip().casefold().replace(" ", "").replace("-", "")
    lookup = {"sfgfp": "sfGFP", "petase": "PETase", "reteplase": "Reteplase"}
    if text not in lookup:
        raise ValueError(f"Unknown/missing protein target: {value!r}; no target guessed from recipe")
    return lookup[text]


def normalize_pocketbase_records(raw_snapshot: Mapping[str, Any], *,
                                 require_complete: bool = True) -> tuple[dict, dict]:
    """Create an analysis-compatible recipe pool from an authorized raw snapshot.

    The output contains only fields verifiable in PocketBase; GP-derived fields are
    *absent or null*. Invalid experimental records fail closed (no silent deletion).
    Nonexperimental records are counted and excluded from the eligible pool.
    """
    if not isinstance(raw_snapshot, Mapping):
        raise TypeError("PocketBase input must be an object")
    meta = raw_snapshot.get("metadata") or {}
    if meta and not isinstance(meta, Mapping):
        raise ValueError("Snapshot metadata must be an object")
    if require_complete and meta.get("designs_possibly_truncated") is True:
        raise ValueError("PocketBase snapshot explicitly marked truncated; use a complete fetch")
    if "collections" in raw_snapshot:
        collections = _json_object(raw_snapshot["collections"], "collections")
        records = collections.get("cfps_designs")
    elif "cfps_designs" in raw_snapshot:
        records = raw_snapshot["cfps_designs"]
    elif "records" in raw_snapshot:
        records = raw_snapshot["records"]
    elif "items" in raw_snapshot:
        records = raw_snapshot["items"]
    else:
        raise ValueError("No cfps_designs collection or records/items array")
    if not isinstance(records, list):
        raise ValueError("cfps_designs must be a list")
    if require_complete and (meta.get("designs_advertised_total_items") is not None):
        expected = int(meta["designs_advertised_total_items"])
        if len(records) < expected:
            raise ValueError(f"Incomplete PocketBase response: {len(records)}/{expected} records")
    designs = {p: [] for p in CANONICAL_PROTEINS}
    ids = set()
    excluded_types = Counter()
    missing_author_count = 0
    for index, record in enumerate(records):
        if not isinstance(record, Mapping):
            raise ValueError(f"Record #{index} is not an object")
        payload = _json_object(record.get("design_json"), f"record[{index}].design_json")
        kind = str(payload.get("sample_type", "")).strip().lower()
        if kind != "experimental":
            excluded_types[kind or "missing"] += 1
            continue
        metadata = _json_object(payload.get("metadata") or {}, f"record[{index}].metadata")
        target = _canonical_target(metadata.get("protein_target"))
        recipe = _json_object(payload.get("reagent_list"), f"record[{index}].reagent_list")
        if not recipe:
            raise ValueError(f"Experimental record #{index} has no reagent_list")
        converted = {}
        total = 0.0
        for k, value in recipe.items():
            if isinstance(value, bool):
                raise ValueError(f"Boolean volume {k} in record #{index}")
            try:
                volume = float(value)
            except (TypeError, ValueError) as exc:
                raise ValueError(f"Non-numeric reagent volume {k} in record #{index}") from exc
            import math
            if not math.isfinite(volume) or volume < 0:
                raise ValueError(f"Invalid reagent volume {k} in record #{index}")
            total += volume
            if volume > 0:
                converted[str(k)] = volume
        if abs(total - 20000.0) > 1e-5:
            raise ValueError(f"Experimental record #{index} total volume {total} nL, expected 20000 nL")
        rid = str(record.get("id") or "").strip()
        if not rid or rid in ids:
            raise ValueError(f"Missing or duplicate immutable record ID: {rid!r}")
        ids.add(rid)
        author = (record.get("author") or record.get("author_sub") or
                  record.get("author_name") or "")
        if not author:
            missing_author_count += 1
        designs[target].append({
            "id": rid,
            "sampleId": str(payload.get("sample_id") or record.get("title") or rid),
            "protein": target,
            "created": record.get("created"),
            "author": str(author),
            "recipeNl": converted,
            # Preserve explicit participant batch metadata for honest completeness checks.
            # Never infer submitted batches from GP-defined clusters or sample ID prefixes.
            "metadata": {key: metadata[key] for key in ("batch_id", "batch_size", "batch_index") if key in metadata},
            # No GP-based properties fabricated; these are neutral placeholders.
            "round1Role": None,
            "sameRecipeAs": None,
            "group": None,
            "forecast": None,
            "vsGinkgoPlates": None,
            "dataOrigin": "PocketBase/cfps_designs",
        })
    if not ids:
        raise ValueError("No valid experimental CFPS records in PocketBase snapshot")
    out = {
        "schema": "cfps-selection-input/v1",
        "built": meta.get("created_utc") or None,
        "input_source": "pocketbase_snapshot",
        "model_based_annotations_available": False,
        "proteins": {p: {"designs": designs[p], "groups": [], "sweeps": [],
                         "totals": {"designs": len(designs[p])}}
                     for p in CANONICAL_PROTEINS},
    }
    summary = {
        "source_type": "pocketbase_snapshot",
        "adapter_version": SOURCE_ADAPTER_VERSION,
        "record_count": len(records),
        "experimental_eligible": len(ids),
        "counts_by_protein": {p: len(designs[p]) for p in CANONICAL_PROTEINS},
        "nonexperimental_excluded": dict(sorted(excluded_types.items())),
        "author_field_missing_count": missing_author_count,
        "gp_baseline_available": False,
        "quantity_semantics": "PocketBase reagent_list stored dispensing volumes (nL)",
        "note": "No GP forecasts/roles were inferred; baseline and hybrid modes unavailable",
    }
    return out, summary


def normalize_analysis_source(path: str | Path, source_type: str) -> tuple[dict, dict]:
    """Read an immutable source file and return compatible data with provenance.

    The caller hashes both source bytes and any normalized cache file separately.
    """
    if source_type not in SOURCE_TYPES:
        raise ValueError(f"source_type must be one of {SOURCE_TYPES}; got {source_type!r}")
    source = Path(path)
    raw = json.loads(source.read_text(encoding="utf-8"))
    if source_type == "pocketbase_snapshot":
        return normalize_pocketbase_records(raw)
    if not isinstance(raw, dict) or not isinstance(raw.get("proteins"), dict):
        raise ValueError("Published snapshot requires top-level proteins mapping")
    for protein in CANONICAL_PROTEINS:
        if protein not in raw["proteins"] or not isinstance(raw["proteins"][protein].get("designs"), list):
            raise ValueError(f"Published snapshot is missing designs for {protein}")
    return raw, {
        "source_type": "published_snapshot", "adapter_version": SOURCE_ADAPTER_VERSION,
        "counts_by_protein": {p: len(raw["proteins"][p]["designs"]) for p in CANONICAL_PROTEINS},
        "gp_baseline_available": True,
        "quantity_semantics": "published recipeNl dispensing volumes (nL)",
        "note": "GP predictions and roles are source-provided, not regenerated",
    }



## 19. Reproducible pre-experimental portfolio analysis

Methods: direct recipe identity, exact external overlap, controlled single-factor
contrasts and sweeps, normalized dispensing-volume/presence features, deterministic
facility-location and Maximin selection. Distances are *not* output measurements,
yield predictions, empirical response correlations, or statistical information gain.

Modes: `gp_model_baseline` reproduces the source's GP-informed roles; `hybrid` keeps
those roles while adding geometric coverage; `model_free` does not consult those roles
or GP forecasts. Only the third mode is available from an unannotated PocketBase pool.

The following implementation is **byte-for-byte identical** to the accompanying
`analysis/cfps_selection/model_free_pipeline.py`, but defined inside this notebook.


In [ ]:
"""Model-free CFPS portfolio selection, exact external overlap, and private visualization export.

Scientific scope: existing submitted recipe IDs, no output GP and no assay outcomes.
Scientific methods and author contributions are documented in docs/CONTRIBUTIONS.md.

Module API mirrored verbatim inside self-contained Colab notebook (not imported there).
Private input/output: never commit recipe-level snapshots, IDs, or overlay JSON publicly.
"""
from __future__ import annotations
from collections import defaultdict, Counter
from dataclasses import dataclass, asdict
import hashlib
import io
import json
import math
import os
from pathlib import Path
from typing import Any, Mapping, Sequence
import zipfile

import numpy as np
from scipy.spatial.distance import cdist
# canonical_recipe, changed_reagents, signature_except and input_sha256 are supplied
# by selection_framework.py in the modular distribution; in Colab notebook,
# their definitions are embedded in a preceding code cell.
if __name__ != '__main__':
    # Import only in the modular package. Colab executes this exact source as
    # a __main__ notebook cell, using the already embedded audit definitions.
    from .selection_framework import canonical_recipe, changed_reagents, signature_except, input_sha256, BASELINE_ROLES

MODEL_FREE_VERSION = '1.1.0'
TARGETS_CANONICAL = ('sfGFP', 'PETase', 'Reteplase')
WATER_FIELDS = frozenset({'nuclease_free_water'})


def protein_target(text: str) -> str:
    """Canonicalize plate's protein name or its template identifier."""
    lower = str(text).casefold()
    if 'reteplase' in lower: return 'Reteplase'
    if 'petase' in lower: return 'PETase'
    if 'sfgfp' in lower or 'gfp' in lower: return 'sfGFP'
    raise ValueError(f'Cannot infer protein target: {text!r}')


def parse_external_plates_zip(path: str | Path) -> dict:
    """Read external team plate submissions, *not* import them into candidate pool.

    Handles individual plate schema and batch schema with nested plates. Experimental
    sample types only. Deduplication is per protein and exact non-water nL signature;
    external replicate factors are not unfolded into fake independent formulations.
    Target inferred from protein_target or recognized plate/sample/file labels.
    Avoids storing external authors in public summaries.
    """
    path = Path(path)
    by_target = {p: defaultdict(list) for p in TARGETS_CANONICAL}
    records_per_target = Counter()
    files = []
    with zipfile.ZipFile(path) as z:
        for info in sorted(z.infolist(),key=lambda x:x.filename):
            name=info.filename
            if info.is_dir() or not name.lower().endswith('.json') or name.startswith('__MACOSX/'):
                continue
            data=json.loads(z.read(info))
            if not isinstance(data,dict): raise ValueError(f'External JSON not a mapping: {name}')
            plates=data.get('plates',[data]); file_count=0
            if not isinstance(plates,list): raise ValueError(f'Unexpected plates object: {name}')
            for plate in plates:
                if not isinstance(plate,dict): raise ValueError(f'Invalid plate: {name}')
                samples=plate.get('samples',[])
                if not isinstance(samples,list): raise ValueError(f'Invalid samples: {name}')
                for sample in samples:
                    if sample.get('sample_type')!='experimental':continue
                    try:
                        target=protein_target(plate.get('protein_target') or plate.get('plate_id') or sample.get('sample_id') or name)
                    except ValueError:
                        target=protein_target(name)
                    recipe=sample.get('reagent_list')
                    if not isinstance(recipe,dict): raise ValueError(f'External sample missing reagent_list: {name}')
                    sig=canonical_recipe(recipe)
                    by_target[target][sig].append({'source_file':Path(name).name,'plate_id':str(plate.get('plate_id','')),'sample_id':str(sample.get('sample_id',''))})
                    file_count+=1
                    records_per_target[target]+=1
            files.append({'file':Path(name).name,'experimental_records':file_count})
    return {'signatures':{p:dict(by_target[p]) for p in TARGETS_CANONICAL},
            'summary':{'experimental_records':{p:int(records_per_target[p]) for p in TARGETS_CANONICAL},
                       'distinct_formulations':{p:len(by_target[p]) for p in TARGETS_CANONICAL},
                       'source_json_files':len(files),'file_counts':files,
                       'overlap_definition':'exact non-water dispensing nL by reagent key and protein, not confirmed chemical equivalence'},
            'sha256':input_sha256(path)}


def external_nearest_distance(data:Mapping[str,Any], external:Mapping[str,Any],
                              target:str,cfg:ModelFreeConfig|None=None)->dict[str,float]:
    """Descriptive near-overlap distance only, NEVER used for automatic exclusion.

    WARNING: assumes shared reagent-key/stock-volume semantics. This is a
    screening metric, not an assertion that different recipes are bioequivalent.
    """
    if external is None:return {}
    rows,_=unique_candidate_pool(data,target)
    others=[{'id':f'external_{i}','recipeNl':dict(sig)}
            for i,sig in enumerate(external['signatures'][target])]
    if not others:return {}
    cfg=cfg or ModelFreeConfig()
    z,_,_=feature_matrix(rows+others,cfg)
    distances=cdist(z[:len(rows)],z[len(rows):],'euclidean')
    return {r['id']:float(v) for r,v in zip(rows,distances.min(axis=1))}


def exact_external_overlap(data: Mapping[str,Any], external: Mapping[str,Any]) -> tuple[dict,dict]:
    """Return private ID-level overlaps + aggregate summaries, by target."""
    if external is None: return ({p:{} for p in TARGETS_CANONICAL}, {'status':'not_supplied'})
    matches={}; summary={}
    for target in TARGETS_CANONICAL:
        look=external['signatures'].get(target,{})
        block=data['proteins'][target]
        results={}
        selected_count=0
        distinct=set()
        for design in block['designs']:
            sig=canonical_recipe(design['recipeNl'])
            if sig in look:
                results[design['id']]=look[sig]
                distinct.add(sig)
                if design.get('round1Role') in BASELINE_ROLES: selected_count+=1
        matches[target]=results
        summary[target]={'participant_records_matching':len(results),
                         'distinct_participant_formulations_matching':len(distinct),
                         'proposed_gp_baseline_records_matching':selected_count,
                         'external_distinct_formulations':len(look)}
    return matches,summary


@dataclass(frozen=True)
class ModelFreeConfig:
    """Frozen configuration; no tuning against predicted yield.

    Quota: working capacity assumption, not official competition validation.
    max_contrast_fraction: maximum *slots* allocated to new complete 3-level
      one-factor sweeps (remaining slots chosen by geometric coverage).
    external_policy: 'flag' or 'exclude' (never auto-purge the underlying pool).
    hybrid: locked GP baseline unique recipes, with optional manual seed IDs.
    model_free: no GP roles or forecasts consulted (except in *separate* comparisons).
    """
    quota: int=320
    external_policy: str='flag'
    objective: str='facility' # facility or maximin
    feature_mode: str='scaled_plus_presence'  # scaled_volume, presence, scaled_plus_presence
    presence_weight: float=0.35
    distance_scale: str='p95'   # p95 or max
    max_contrast_fraction: float=0.20
    max_sweep_share_of_new_slots: float=0.50
    max_families_per_factor: int=5
    max_levels_per_sweep: int=5
    min_levels_per_sweep: int=3
    factor_bins: int=5
    near_external_threshold: float=0.20  # scaled-distance HEURISTIC; annotate only
    n_projection_components: int=2

    def validate(self):
        if self.quota<1: raise ValueError('quota must be positive')
        if self.external_policy not in {'flag','exclude'}: raise ValueError('external_policy must be flag|exclude')
        if self.objective not in {'facility','maximin'}: raise ValueError('objective must be facility|maximin')
        if self.feature_mode not in {'scaled_volume','presence','scaled_plus_presence'}: raise ValueError('Bad feature_mode')
        if self.distance_scale not in {'p95','max'}: raise ValueError('Bad distance_scale')
        if not 0<=self.max_contrast_fraction<=1: raise ValueError('Bad contrast fraction')
        if not 0<=self.max_sweep_share_of_new_slots<=1: raise ValueError('Bad new-slot sweep share')
        if self.presence_weight<0: raise ValueError('Bad presence weight')
        if self.max_families_per_factor<0 or self.max_levels_per_sweep<2: raise ValueError('Bad sweep caps')
        if self.min_levels_per_sweep<2 or self.min_levels_per_sweep>self.max_levels_per_sweep: raise ValueError('Bad min sweep levels')
        if self.factor_bins<2: raise ValueError('Bad factor bins')
        if self.near_external_threshold<0: raise ValueError('Bad near-external threshold')


def unique_candidate_pool(data: Mapping[str,Any],target:str, *, forbidden_ids: set[str] | None=None) -> tuple[list[dict],dict]:
    """Distinct input recipes with ID provenance, independent of GP-derived grouping.

    Representative ID is lexical minimum; the origin's round1Role is stored for
    comparison only and is never consulted for the *model_free* selection.
    """
    forbidden_ids=forbidden_ids or set()
    unique=defaultdict(list)
    for raw in data['proteins'][target]['designs']:
        key=canonical_recipe(raw['recipeNl']);unique[key].append(raw)
    rows=[]; all_records=[]
    for key, originals in sorted(unique.items(),key=lambda x: min(r['id'] for r in x[1])):
        ids=sorted(r['id'] for r in originals)
        row={'id':ids[0], 'ids':ids, 'recipeNl':dict(key),
             'author_count':len(set(str(r.get('author','')) for r in originals)),
             'external_overlap':bool(any(rid in forbidden_ids for rid in ids)),
             'baseline_ids':[r['id'] for r in originals if r.get('round1Role') in BASELINE_ROLES],
             'group_labels':sorted({r['group']['letter'] for r in originals if r.get('group')}),
             'protein':target}
        rows.append(row)
        all_records.extend(ids)
    return rows, {'records':len(all_records),'distinct':len(rows),'duplicate_excess':len(all_records)-len(rows)}


def feature_matrix(rows:list[dict],cfg:ModelFreeConfig) -> tuple[np.ndarray,list[str],dict]:
    """Explicit recipe volume feature map; no stock molarities inferred.

    Per-reagent scaling by positive p95 (or observed maximum), capped at 1 for
    larger values to limit outliers. Presence indicator adds orthogonal weight.
    Water is excluded, and features constant across target are removed.
    """
    cfg.validate()
    factors=sorted({k for r in rows for k in r['recipeNl'] if k not in WATER_FIELDS})
    if not factors:raise ValueError('No chemistry factors')
    A=np.array([[float(r['recipeNl'].get(k,0)) for k in factors] for r in rows],float)
    variable=np.ptp(A,axis=0)>1e-9
    A=A[:,variable]; factors=[f for f,b in zip(factors,variable) if b]
    if not len(factors):raise ValueError('No variable chemistry factors')
    scale=np.array([np.quantile(A[A[:,j]>0,j],0.95) if cfg.distance_scale=='p95' else max(A[:,j]) for j in range(A.shape[1])],float)
    scale=np.maximum(scale,1e-12)
    volume=np.minimum(A/scale,1.0)
    presence=(A>0).astype(float)
    if cfg.feature_mode=='scaled_volume':Z=volume; names=factors
    elif cfg.feature_mode=='presence':Z=presence; names=['presence:'+k for k in factors]
    else:
        Z=np.hstack([volume,math.sqrt(cfg.presence_weight)*presence])
        names=factors+['presence:'+k for k in factors]
    meta={'factors':factors,'scaling':dict(zip(factors,scale.tolist())),
          'distance_unit':'scaled nL per reagent (within-stock), NOT molarity',
          'n_features':int(Z.shape[1]),'exclude_balancers':sorted(WATER_FIELDS)}
    return Z,names,meta


def discover_exact_sweeps(rows:list[dict],cfg:ModelFreeConfig)->list[dict]:
    """One-factor sweeps discovered on *candidate pool* (not modeled trajectories).

    Signature groups are identical on all non-water, non-factor reagents.
    A family requires >= min_levels_per_sweep distinct factor values; it may
    overlap with another family and so cannot be counted as an independent DoE.
    """
    cfg.validate()
    all_factors=sorted({k for r in rows for k in r['recipeNl'] if k not in WATER_FIELDS})
    families=[]
    for factor in all_factors:
        grouped=defaultdict(list)
        for i,row in enumerate(rows):
            grouped[signature_except(row['recipeNl'],factor)].append(i)
        for background,idxs in grouped.items():
            levels=sorted({float(rows[i]['recipeNl'].get(factor,0)) for i in idxs})
            if len(levels)<cfg.min_levels_per_sweep: continue
            by_level=defaultdict(list)
            for i in idxs:by_level[float(rows[i]['recipeNl'].get(factor,0))].append(rows[i]['id'])
            chosen_ids=[min(by_level[level]) for level in levels]
            families.append({'factor':factor,'levels':levels,'ids':chosen_ids,
                             'n_levels':len(levels),'span_nL':levels[-1]-levels[0],
                             'background_hash':hashlib.sha256(repr(background).encode()).hexdigest()[:16]})
    return sorted(families,key=lambda f:(-f['n_levels'],-f['span_nL'],f['factor'],f['background_hash']))


def _sweep_subset(family:dict, max_levels:int) ->list[str]:
    """Pick range extremes and spread intermediate indices deterministically."""
    n=len(family['ids']);k=min(n,max_levels)
    if k==n:return family['ids'][:]
    # Index fractions might collide under rounding: fill from left if so.
    indices=sorted(set(int(round(i*(n-1)/(k-1))) for i in range(k)))
    for i in range(n):
        if len(indices)>=k:break
        if i not in indices:indices.append(i)
    return [family['ids'][i] for i in sorted(indices)]


def _select_sweeps(rows:list[dict],start:set[str],capacity:int,cfg:ModelFreeConfig,forbidden:set[str])->tuple[set[str],list[dict]]:
    """Reserve limited preexisting exact 3+level sweeps, no model statistics."""
    families=discover_exact_sweeps(rows,cfg)
    allowed={r['id'] for r in rows}-forbidden
    selected=set(start); taken=[];factor_count=Counter()
    free_slots=max(0,capacity-len(start))
    sweep_extra_limit=min(free_slots,int(math.floor(cfg.quota*cfg.max_contrast_fraction)),
                          int(math.floor(free_slots*cfg.max_sweep_share_of_new_slots)))
    spent=0
    # Ordering spans factor families; per-factor cap prevents a single dominant ingredient.
    for fam in families:
        factor=fam['factor']
        if factor_count[factor]>=cfg.max_families_per_factor: continue
        ids=_sweep_subset(fam,cfg.max_levels_per_sweep)
        if any(rid not in allowed for rid in ids):continue
        extra=[rid for rid in ids if rid not in selected]
        if len(extra)+spent>sweep_extra_limit or len(selected)+len(extra)>capacity:continue
        if not extra: continue
        selected.update(extra); spent+=len(extra)
        factor_count[factor]+=1
        taken.append({'factor':factor,'ids':ids,'levels_nL':fam['levels'],
                      'new_ids':extra,'n_new':len(extra),'background_hash':fam['background_hash']})
    return selected,taken


def _distance_and_select(rows:list[dict], selected_ids:set[str],cfg:ModelFreeConfig,quota:int,allowed_indices:set[int]) ->tuple[set[str],np.ndarray,np.ndarray,dict]:
    Z,names,meta=feature_matrix(rows,cfg)
    D=cdist(Z,Z,'euclidean')
    id_idx={r['id']:i for i,r in enumerate(rows)}
    selected=set(selected_ids)
    indices=[id_idx[i] for i in selected]
    nearest=D[:,indices].min(axis=1) if indices else np.full(len(rows),float('inf'))
    W=np.full(len(rows),1/len(rows),float)
    while len(selected)<quota and allowed_indices:
        ix=np.array(sorted(allowed_indices),dtype=int)
        if not indices:
            total=W@D
            pick=int(ix[np.argmin(total[ix])]) if cfg.objective=='facility' else int(ix[np.argmax(total[ix])])
        elif cfg.objective=='maximin':
            scores=nearest[ix]
            pick=int(ix[np.argmax(scores)])
        else:
            # marginal reduction in weighted sum of distance to nearest representative
            improvements=np.maximum(nearest[:,None]-D[:,ix],0)
            gains=W@improvements
            pick=int(ix[np.argmax(gains)])
        selected.add(rows[pick]['id']);indices.append(pick)
        nearest=np.minimum(nearest,D[:,pick]);allowed_indices.remove(pick)
    return selected,Z,D,meta


def _factor_bins(rows:list[dict], selected_ids:set[str],n_bins:int)->tuple[dict,dict]:
    factors=sorted({k for r in rows for k in r['recipeNl'] if k not in WATER_FIELDS})
    cov={}; enc={}
    for f in factors:
        values=np.array([float(r['recipeNl'].get(f,0)) for r in rows])
        positive=np.unique(values[values>0])
        if len(positive)==0: continue
        bounds=np.unique(np.quantile(positive,np.linspace(0,1,n_bins+1)))
        # Bin 0 is absence; positive bins are +1.. ; ties robustly collapsed.
        assigned=np.where(values==0,0,1+np.searchsorted(bounds[1:-1],values,side='right'))
        enc[f]=assigned
        active={int(assigned[i]) for i,r in enumerate(rows) if r['id'] in selected_ids}
        possible={int(x) for x in assigned}
        cov[f]={'represented_bins':len(active), 'possible_bins':len(possible),
                'coverage_fraction':float(len(active)/len(possible)),
                'missing_bins':sorted(possible-active),
                'has_zero_in_candidates':0 in possible}
    return cov,enc


def _pair_bin_coverage(encoded:dict,rows:list[dict],chosen:set[str],count_top:int=8)->dict:
    variability=sorted(encoded,key=lambda f:(-len(set(encoded[f])),f))[:count_top]
    selidx={i for i,r in enumerate(rows) if r['id'] in chosen}
    output={}
    for j,a in enumerate(variability):
        for b in variability[j+1:]:
            combos=set(zip(encoded[a].tolist(),encoded[b].tolist()))
            have={(int(encoded[a][i]),int(encoded[b][i])) for i in selidx}
            output[f'{a} × {b}']={'occupied_candidate_bins':len(combos),
                                  'represented_bins':len(have),
                                  'coverage_fraction':float(len(have)/len(combos))}
    return output


def exact_selected_contrast_summary(rows:list[dict], chosen:set[str],max_examples:int=36)->dict:
    """Find exact one-factor pairs among the chosen physical formulations.

    This counts pairs within existing recipe backgrounds, not independent degrees
    of freedom. Reports sampled examples only to bound browser payload size.
    """
    subset=[r for r in rows if r['id'] in chosen]
    factors=sorted({k for r in subset for k in r['recipeNl'] if k not in WATER_FIELDS})
    pairs=set();per_factor=Counter(); examples=[]
    for factor in factors:
        groups=defaultdict(list)
        for r in subset:
            groups[signature_except(r['recipeNl'],factor)].append(r)
        for members in groups.values():
            for i in range(len(members)):
                for j in range(i+1,len(members)):
                    a,b=members[i],members[j]
                    if math.isclose(float(a['recipeNl'].get(factor,0)),float(b['recipeNl'].get(factor,0)),abs_tol=1e-8):continue
                    pair=tuple(sorted([a['id'],b['id']]))
                    if pair in pairs:continue
                    pairs.add(pair);per_factor[factor]+=1
                    if len(examples)<max_examples:
                        examples.append({'a':pair[0],'b':pair[1],'factor':factor,
                                'levels_nL':sorted([float(a['recipeNl'].get(factor,0)),float(b['recipeNl'].get(factor,0))])})
    return {'unique_exact_pairs':len(pairs),'factor_counts':dict(sorted(per_factor.items())),
            'examples':examples,
            'note':'Pairs sharing families or ingredients are not statistically independent contrasts'}


def _pca_projection(Z:np.ndarray)->np.ndarray:
    """Deterministic PCA coordinates as a display ONLY; distances use full feature matrix."""
    C=Z-Z.mean(axis=0)
    if min(C.shape)<2:
        result=np.zeros((len(Z),2));result[:,0]=C[:,0] if C.shape[1] else 0
        return result
    _u,s,vt=np.linalg.svd(C,full_matrices=False)
    arr=C@vt[:2].T
    for j in range(2):
        pivot=int(np.argmax(abs(vt[j])))
        if vt[j,pivot]<0: arr[:,j]*=-1
    return arr


def _quantiles(nearest:np.ndarray)->dict:
    return {'mean':float(np.mean(nearest)),'p90':float(np.quantile(nearest,.9)),
            'max':float(np.max(nearest))}


def analyze_portfolio(data:Mapping[str,Any],target:str,*,config:ModelFreeConfig=ModelFreeConfig(),
                      mode:str='model_free',external_matches:dict|None=None,
                      seeds:Sequence[str]=(), include_geometry:bool=True,
                      external_nearest_by_id:Mapping[str,float]|None=None)->dict:
    """Full deterministic model-free/hybrid portfolio for existing submissions.

    * gp_model_baseline = read-only reconstruction of the published GP-informed selection.
    * hybrid = lock that baseline and use model-free sweeps/coverage to fill gaps.
    * model_free = no forecast/GP link roles consulted during selection.
    External exact overlaps may be flagged or excluded; locked external overlaps
    in hybrid are an error when exclude chosen (never silently violate baseline).
    """
    config.validate()
    if mode not in {'gp_model_baseline','hybrid','model_free'}: raise ValueError(f'Invalid mode {mode}')
    if target not in data['proteins']:raise ValueError('Unknown target')
    external_matches=external_matches or {}
    external_nearest_by_id=external_nearest_by_id or {}
    rows,pool_stats=unique_candidate_pool(data,target,forbidden_ids=set(external_matches))
    if config.quota>len(rows):raise ValueError(f'Quota {config.quota} exceeds distinct pool {len(rows)}')
    map_alias={ident:r['id'] for r in rows for ident in r['ids']}
    forbid={r['id'] for r in rows if r['external_overlap']} if config.external_policy=='exclude' else set()
    locked={map_alias[i] for i in seeds}
    if mode in {'gp_model_baseline','hybrid'}:
        locked.update(r['id'] for r in rows if r['baseline_ids'])
    if unknown := set(seeds)-set(map_alias):raise ValueError(f'Unknown seed IDs: {sorted(unknown)[:5]}')
    if len(locked)>config.quota:raise ValueError('Locked baseline larger than quota')
    if conflicted := locked & forbid:
        raise ValueError(f'GP baseline/manual seeds overlap external data under exclude policy ({len(conflicted)}). Use flag mode or resolve explicitly.')
    selected=set(locked);chosen_sweeps=[]
    if mode!='gp_model_baseline':
        selected,chosen_sweeps=_select_sweeps(rows,selected,config.quota,config,forbid)
    allowed={i for i,r in enumerate(rows) if r['id'] not in selected|forbid}
    # GP baseline is reconstructed only, never secretly padded.
    if mode=='gp_model_baseline':
        Z,fn,meta=feature_matrix(rows,config);D=cdist(Z,Z)
    else:
        selected,Z,D,meta=_distance_and_select(rows,selected,config,config.quota,allowed)
    id_idx={r['id']:i for i,r in enumerate(rows)}
    idx=[id_idx[i] for i in sorted(selected)]
    nearest=D[:,idx].min(axis=1) if idx else np.full(len(rows),float('nan'))
    factor_cov,bins=_factor_bins(rows,selected,config.factor_bins)
    pair_cov=_pair_bin_coverage(bins,rows,selected)
    projection=_pca_projection(Z) if include_geometry else np.zeros((len(rows),2))
    contrasts=exact_selected_contrast_summary(rows,selected)
    n_external_selected=sum(r['external_overlap'] and r['id'] in selected for r in rows)
    authors=set(); selected_authors=set()
    for d in data['proteins'][target]['designs']:
        authors.add(str(d.get('author','')))
        if map_alias[d['id']] in selected: selected_authors.add(str(d.get('author','')))
    return {
        'schema':'cfps-model-free-portfolio/v1','algorithm_version':MODEL_FREE_VERSION,
        'target':target,'mode':mode,'selection_status':'PROVISIONAL: pending manual controls, rules, lab feasibility and group decision',
        'config':asdict(config),'selected_ids':sorted(selected),
        'selected_all_submission_ids':sorted(d['id'] for d in data['proteins'][target]['designs'] if map_alias[d['id']] in selected),
        'n_selected':len(selected),'gap_to_working_quota':config.quota-len(selected),
        'locked_count':len(locked),'sweep_reserved_count':sum(x['n_new'] for x in chosen_sweeps),
        'selected_sweeps':chosen_sweeps,'external_overlaps_selected':n_external_selected,
        'external_overlaps_pool':sum(r['external_overlap'] for r in rows),
        'near_external_pool_count':sum(external_nearest_by_id.get(r['id'],float('inf'))<=config.near_external_threshold for r in rows),
        'near_external_selected_count':sum(external_nearest_by_id.get(r['id'],float('inf'))<=config.near_external_threshold for r in rows if r['id'] in selected),
        'pool':pool_stats,'authors_represented':len(selected_authors),'authors_total':len(authors),
        'exact_selected_contrasts':contrasts,
        'distance':_quantiles(nearest),'dimension_count':meta['n_features'],
        'feature_meta':meta,
        'factor_bin_coverage_mean':float(np.mean([v['coverage_fraction'] for v in factor_cov.values()])) if factor_cov else None,
        'factor_bins':factor_cov,'pair_bin_coverage_mean':float(np.mean([v['coverage_fraction'] for v in pair_cov.values()])) if pair_cov else None,
        'pair_bins':pair_cov,
        'visual_rows':[{'id':r['id'],'ids':r['ids'], 'selected':r['id'] in selected,
                        'externalOverlap':r['external_overlap'],
                        'externalNearestDistance':round(float(external_nearest_by_id[r['id']]),6) if r['id'] in external_nearest_by_id else None,
                        'point':[round(float(projection[i,j]),6) for j in range(2)],
                        'groups':r['group_labels'],
                        'nReagents':len(r['recipeNl'])}
                       for i,r in enumerate(rows)]
    }


def write_private_analysis_payload(results:dict, out:str|Path, *, input_paths:Sequence[str|Path],
                                   external_summary:dict|None=None, provenance:dict|None=None)->dict:
    """Write *private* JSON; never place in static/ or Git. Contains participant IDs."""
    path=Path(out);path.parent.mkdir(parents=True,exist_ok=True)
    body={'schema':'cfps-model-free-visualization/v1','privacy':'PRIVATE: authenticated participants only',
          'algorithm_version':MODEL_FREE_VERSION,'input_hashes':{Path(p).name:input_sha256(p) for p in input_paths},
          'targets':results,'external_summary':external_summary,
          'provenance':provenance or {},
          'caveat':'PCA projection is visual only; model-free coverage is not biological efficacy'}
    path.write_text(json.dumps(body,sort_keys=True,indent=2,allow_nan=False)+'\n',encoding='utf-8')
    try:os.chmod(path,0o600)
    except OSError:pass
    return {'path':str(path),'sha256':input_sha256(path),'size_bytes':path.stat().st_size}


def model_free_run(data_path:str|Path, external_zip:str|Path|None=None, *,
                   quota:int=320, external_policy:str='flag',seed_ids_by_target:dict|None=None,
                   objectives:Sequence[str]=('facility','maximin'),
                   modes:Sequence[str]=('gp_model_baseline','hybrid','model_free')) ->dict:
    """Top-level reproducible pipeline: read inputs, external audit, run strategy grid.

    No network, no credential use, no row-level public output.
    """
    data_path=Path(data_path)
    data=json.loads(data_path.read_text(encoding='utf-8'))
    if data.get('model_based_annotations_available') is False and any(m != 'model_free' for m in modes):
        raise ValueError('PocketBase-only input has no GP-informed baseline or run-role annotations. ' +
                         'Use modes=("model_free",), or analyze a published GP-annotated snapshot separately.')
    external=parse_external_plates_zip(external_zip) if external_zip else None
    matches,summary=exact_external_overlap(data,external)
    results={}
    for target in TARGETS_CANONICAL:
        t_results={}
        ext_dist=external_nearest_distance(data,external,target) if external else {}
        for mode in modes:
            for obj in ('facility',) if mode=='gp_model_baseline' else objectives:
                config=ModelFreeConfig(quota=quota,external_policy=external_policy,objective=obj)
                output=analyze_portfolio(data,target,config=config,mode=mode,
                                         external_matches=matches.get(target,{}),
                                         seeds=(seed_ids_by_target or {}).get(target,()),
                                         external_nearest_by_id=ext_dist)
                t_results[f'{mode}:{obj}']=output
        results[target]=t_results
    return {'data':data,'external':external,'matches':matches,'external_summary':summary,'results':results}


In [ ]:

# 20. Resolve the explicitly selected input, normalize where necessary, and run.
from pathlib import Path
import json, hashlib
from IPython.display import display
import pandas as pd
if RUN_MODEL_FREE_PIPELINE:
    if INPUT_SOURCE == 'published_snapshot':
        raw_source_path = Path(PUBLISHED_SNAPSHOT_PATH).expanduser()
        normalized_kind = 'published_snapshot'
    elif INPUT_SOURCE == 'pocketbase_snapshot':
        raw_source_path = Path(POCKETBASE_SNAPSHOT_PATH).expanduser()
        normalized_kind = 'pocketbase_snapshot'
    elif INPUT_SOURCE == 'pocketbase_live':
        if not RUN_POCKETBASE_FETCH or 'RAW_SNAPSHOT_PATH' not in globals():
            raise RuntimeError('Live PocketBase fetch did not produce RAW_SNAPSHOT_PATH')
        raw_source_path = Path(RAW_SNAPSHOT_PATH)
        normalized_kind = 'pocketbase_snapshot'
    else:
        raise ValueError('INPUT_SOURCE must be published_snapshot, pocketbase_snapshot or pocketbase_live')
    if not raw_source_path.is_file():
        raise FileNotFoundError(f'Missing private source file: {raw_source_path}')

    data, source_provenance = normalize_analysis_source(raw_source_path, normalized_kind)
    private_folder = PRIVATE_ROOT / 'portfolio_analysis'
    private_folder.mkdir(parents=True, exist_ok=True)
    normalized_input = private_folder / 'normalized_candidate_pool_PRIVATE.json'
    normalized_input.write_text(json.dumps(data, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    normalized_input.chmod(0o600)
    print('Source summary:', source_provenance)
    if normalized_kind == 'published_snapshot':
        analysis_input = raw_source_path
        analysis_modes = ('gp_model_baseline', 'hybrid', 'model_free')
    else:
        analysis_input = normalized_input
        analysis_modes = ('model_free',)
    ext = Path(EXTERNAL_PLATES_ZIP).expanduser()
    ext = ext if ext.is_file() else None
    if ext is None:
        print('External plates not supplied: external exact-overlap audit disabled.')

    run = model_free_run(analysis_input, ext, quota=WORKING_QUOTA_NOTEBOOK,
                         external_policy=EXTERNAL_POLICY, modes=analysis_modes)
    original_paths = [raw_source_path] + ([ext] if ext else [])
    source_manifest = {
        'input_source': INPUT_SOURCE,
        'adapter_version': SOURCE_ADAPTER_VERSION,
        'original_input_sha256': input_sha256(raw_source_path),
        'normalized_input_sha256': input_sha256(normalized_input),
        'input_counts': source_provenance['counts_by_protein'],
        'gp_baseline_available': source_provenance['gp_baseline_available'],
        'external_input_sha256': input_sha256(ext) if ext else None,
        'working_quota_not_verified': WORKING_QUOTA_NOTEBOOK,
        'external_policy': EXTERNAL_POLICY,
        'algorithm_version': MODEL_FREE_VERSION,
    }
    (private_folder / 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2, sort_keys=True) + '\n')
    private_payload = write_private_analysis_payload(
        run['results'], private_folder / 'model_free_analysis.json',
        input_paths=original_paths, external_summary=run['external_summary'],
        provenance=source_manifest)
    table = pd.DataFrame([{
       'protein': t, 'strategy': key, 'selected': entry['n_selected'],
       'mean_nearest_distance': round(entry['distance']['mean'], 6),
       'factor_bin_coverage': round(entry['factor_bin_coverage_mean'], 4) if entry['factor_bin_coverage_mean'] is not None else None,
       'exact_contrast_pairs': entry['exact_selected_contrasts']['unique_exact_pairs'],
       'external_exact_matches': entry['external_overlaps_selected']
    } for t, strategies in run['results'].items() for key, entry in strategies.items()])
    display(table)
    table.to_csv(private_folder / 'strategy_comparison.csv', index=False)
    (private_folder / 'selected_ids_PRIVATE.json').write_text(json.dumps({
       t: {strategy: portfolio['selected_ids'] for strategy, portfolio in group.items()}
       for t, group in run['results'].items()}, indent=2, sort_keys=True) + '\n')
    print('Protected visualization JSON:', private_payload['path'])


In [ ]:

# 21. Export the reproducibility bundle for comparison with another run.
# Contains participant IDs, external-overlap flags and potentially sensitive
# predictions: do not commit or post the archive publicly.
if RUN_MODEL_FREE_PIPELINE:
    from zipfile import ZipFile, ZIP_DEFLATED
    from datetime import datetime, timezone
    export_path = PRIVATE_ROOT / ('cfps_private_run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.zip')
    export_files = sorted(p for p in private_folder.rglob('*') if p.is_file())
    with ZipFile(export_path, 'w', ZIP_DEFLATED) as z:
        for item in export_files:
            z.write(item, arcname=item.relative_to(private_folder))
    with ZipFile(export_path) as z:
        assert z.testzip() is None, 'Corrupt results ZIP'
    print('Download this PRIVATE results archive for cross-run comparison:', export_path)
    print('ZIP SHA256:', input_sha256(export_path))
    if AUTO_DOWNLOAD_PRIVATE_RESULTS:
        try:
            from google.colab import files as colab_files
            colab_files.download(str(export_path))
        except ImportError:
            print('Outside Colab: the private ZIP remains at the path printed above.')



## 22. Deployment and evidence boundaries

- The source distribution contains no private participant records or tokens.
- Transfer `model_free_analysis.json` exclusively to a Keycloak-gated, Git-ignored
  deployment directory. Do **not** place it under `static/` or commit it.
- The optional `clusters.json.gz` is a *published-link diagnostic*, not an empirical
  response covariance matrix; its absence does not affect model-free selection.
- PocketBase requires valid authorized credentials in the live mode and may expose a
  different, later participant snapshot than the published GP-informed export.
- Run identifiers, exact input checksums and population counts are saved for comparison.
- All selection results are provisional until final quota, controls and eligibility
  constraints are independently verified.
